# Topological Quantum Transistor: main ablation study

This notebook is the primary computational study for **Topological Quantum Transistor: A Standardized, Braid-Compiled Primitive for Quantum Feature Learning**, by Javier Villalba-Díez and Joaquín Ordieres-Meré.

The 12 matched arms distinguish the audited notebook matrix model (TQT-NB), exact controls, and a separately repaired logical TQT/QT pair. The earlier differentiable QT study is archived in `legacy/`; its scores are not matched controls for this experiment.

**Before running:** start the kernel in the repository root, supply the original Parquet tensors in `data/outs/` (or change `CONFIG["data_dir"]`), and read [the reproduction guide](docs/REPRODUCIBILITY.md). Executing the next cell starts the run. `preflight_only=True` audits the real data without fitting the feature models. `mode="smoke"` uses synthetic data and cannot produce manuscript evidence.

**Recorded outputs:** the saved outputs below are retained from the supplied completed run, identified as `run_0f4dee9df4ca` and logged on 2026-09-07. They were not recomputed for this repository documentation update. Their absolute macOS paths identify the original execution environment; current default paths are relative. Only the introductory documentation and input/output defaults have been changed. The scientific implementation and saved outputs are preserved.

See [Supplementary Material](docs/SUPPLEMENTARY_MATERIAL.md) for configuration, arm definitions, provenance and interpretation. Raw data, subject mappings, and the original external CSV/JSON/checkpoint/report files are not bundled.


In [4]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""R1.M.1 standalone ablations: duplicate-safe, ABI-isolated edition.

RUN IN JUPYTER from the repository root. Configure CONFIG before executing
this cell; it starts the study immediately. Default input: data/outs.
Default output: outputs/R1_M1_ablation. See docs/REPRODUCIBILITY.md.

This revision fixes the reported 'Identical tensors have contradictory labels'
exception without deleting records, changing labels, or pretending conflicts
were resolved. Only .parquet tensors are loaded; Left/Right .pgm previews and
hidden macOS metadata are ignored. '-walking' is class 1; '-not_walking' is
class 0. Legacy '-standing' and negative spelling aliases are supported.

DUPLICATE POLICY (applies equally to every arm):
* Hash tensor SHAPE and canonical numeric values, not bytes without a shape.
* Retain every loaded tensor with its original filename label, even where
  identical tensors have different labels. Never majority-relabel or drop them.
* Join identical-tensor records with their original filename/subject groups
  using connected components. A label-independent common-prefix identity also
  joins potential collisions from the existing training-width crop operation.
  This prefix is used ONLY to constrain splits; fitted preprocessing still
  selects width and fits normalization/PLS/PCA on training partitions only.
* Joined groups cannot cross any outer or inner train/evaluation boundary.
  The same effective groups are the units for descriptive paired bootstrapping.
* Export exact duplicates, prefix-only collisions, original/effective group IDs,
  label conflicts, split decisions, and retained class counts before training.
* Check ALL nested folds and front-end fits before compiling/training TQT.
  Try deterministic alternative stratified group splits for feasibility only.
  The requested 3 x 3 folds are preferred. Any reduction (minimum 2 x 2) is
  explicitly reported. Never fall back to row-wise splitting.
* If the data cannot support valid grouped nested evaluation, return an audit
  report with status='diagnostic_only', not a traceback or fabricated scores.

NUMERICAL SCOPE:
The same 12 ablation arms, TQT matrices, compiled/exact controls, CEM budgets,
and balanced logistic heads are retained. A finite 16-column PLS/PCA output
with zero-variance columns is recorded as rank-deficient, not treated as a
software failure. No new informative components are invented or substituted.
Group IDs derived from 'segment_###' are NOT verified subject identifiers.
Use groups_csv only when an actual subject/session map is available.

DEPENDENCIES:
Scientific imports occur in a checked child process, never in the notebook
kernel. A healthy cached environment is reused. When required, a dedicated
venv with pinned packages is created without changing Anaconda. First-time
recovery needs Internet or RUNTIME['wheelhouse']. The runner is synchronous;
interrupting it stops the child. Existing successful checkpoints are resumable,
but checkpoints from earlier scientific/data-policy versions are not reused.

OUTPUT:
Vector PDF and SVG, 300-dpi PNG, full PDF report, CSVs, LaTeX, and checkpoints.
'R1_M1_data_quality_report.pdf', 'duplicate_tensor_audit.csv',
'dataset_manifest.csv' and 'data_loading_audit.json' are written before fitting.
No historical or synthetic accuracies are substituted for study results.
Smoke mode is explicitly watermarked and must not be submitted as evidence.
"""
from __future__ import annotations

# STANDARD LIBRARY ONLY in the parent process -- deliberately no numpy/sklearn.
import ast
import copy
import hashlib
import inspect
import json
import math
import os
import platform
import signal
import subprocess
import sys
import textwrap
import time
import uuid
from collections import deque
from pathlib import Path

# ---------------------------------------------------------------------------
# USER SETTINGS. Paths are relative to the notebook kernel working directory.
# ---------------------------------------------------------------------------
CONFIG = dict(
    data_dir="data/outs",
    output_dir="outputs/R1_M1_ablation",
    notebook_path=None,             # None: auto-find original or use embedded core.
    groups_csv=None,                # Recommended: CSV columns filename,group.
    group_level="filename_group",  # Use 'subject' ONLY with a genuine subject map.
    outer_folds=3,
    inner_folds=3,                  # Group-disjoint nested CV, not random rows.
    preflight_only=False,          # True: audit real data/splits/front ends without TQT fits.
    split_search_attempts=32,       # Feasibility only; never chosen using model accuracy.
    allow_fold_reduction=True,     # Explicitly logged; never below 2 outer / 2 inner folds.
    outer_seed=42,
    inner_seed=123,
    n_components=16,                # Intentionally fixed; no silent dimension drop.
    projection_seed=23,             # Notebook's 11 + epsilon at epsilon=12.
    epsilon=12,                    # Fixed, historical operating point; NOT retuned.
    codebook=(0.5, 1.0, 1.5, 2.0),
    fixed_alpha=(0.8, 0.8, 0.8),     # Notebook CEM starting point, not test-picked.
    run_seeds=(54, 1054, 2054),      # Paired optimization/random-feature repeats.
    cem_epochs=20,
    cem_popsize=10,                 # 10 mirrored pairs = 20 candidates/epoch.
    cem_sigma=0.3,
    cem_decay=0.97,
    cem_head_C=1.0,                 # Common C while searching the 3 scales.
    head_C_grid=(0.01, 0.1, 1.0, 10.0, 100.0),
    standardize_head=True,          # Training-only, identically applied to all LR.
    selection_metric="accuracy",   # 'accuracy' or 'balanced_accuracy'.
    lr_max_iter=4000,
    compiler_seed=20260907,
    compiler_dictionary_depth=8,    # Preserves source dictionary lengths 0,...,7.
    compiler_trials=1024,           # Preserves source successful-search budget.
    angle_quantum=float(math.pi/128),
    include_pca_tqt=True,           # Completes PLS/PCA x LR/TQT comparison.
    include_exact_grid_control=True,# Separates angle binning from braid effects.
    include_logical_pair=True,      # Repaired TQT vs genuine logical-CRY QT.
    bootstrap_replicates=2000,
    bootstrap_seed=7319,
    dpi=300,
    resume=True,
    show_figures=False,
    blas_threads=1,                 # Small matrices; avoids thread oversubscription.
    mode="study",                  # 'smoke' is synthetic, never manuscript data.
)

# Runtime controls are separate from scientific hyperparameters.
RUNTIME = dict(
    environment="auto",          # 'auto', 'isolated', or 'current'.
    cache_dir=str(Path.home()/".cache"/"tqt_r1m1"),
    allow_install=True,           # ONLY into this script's dedicated venv.
    wheelhouse=None,              # Optional offline folder of compatible wheels.
    python_executable=None,       # None uses sys.executable; do not change normally.
    probe_timeout=120,
)

# One coherent binary stack; not a claim that these are the latest releases.
# All direct/transitive analysis dependencies are pinned in managed recovery.
PINNED_REQUIREMENTS = (
    "numpy==2.2.6", "scipy==1.15.3", "scikit-learn==1.6.1",
    "pandas==2.2.3", "pyarrow==20.0.0", "matplotlib==3.10.3",
    "threadpoolctl==3.6.0", "joblib==1.4.2", "contourpy==1.3.2",
    "cycler==0.12.1", "fonttools==4.58.0", "kiwisolver==1.4.8",
    "packaging==25.0", "pillow==11.2.1", "pyparsing==3.2.3",
    "python-dateutil==2.9.0.post0", "pytz==2025.2", "tzdata==2025.2",
    "six==1.17.0",
)


def _json_fallback(obj):
    if isinstance(obj, os.PathLike):
        return os.fspath(obj)
    if hasattr(obj, "item"):
        return obj.item()
    if hasattr(obj, "tolist"):
        return obj.tolist()
    raise TypeError(f"Not JSON serializable: {type(obj).__name__}")


def _save_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name+"."+uuid.uuid4().hex+".tmp")
    temporary.write_text(json.dumps(obj, indent=2, default=_json_fallback,
                                    allow_nan=False), encoding="utf-8")
    temporary.replace(path)


def _process_env(cache):
    env = os.environ.copy()
    # Neither Python imports nor pip installation targets may leak from Anaconda.
    for key in ("PYTHONPATH", "PYTHONHOME", "PYTHONSTARTUP", "PYTHONUSERBASE",
                "PIP_TARGET", "PIP_PREFIX", "PIP_USER", "PIP_REQUIREMENT",
                "PIP_CONSTRAINT", "PIP_EXTRA_INDEX_URL", "PIP_FIND_LINKS"):
        env.pop(key, None)
    env.update(PYTHONNOUSERSITE="1", PYTHONUNBUFFERED="1", MPLBACKEND="Agg",
               OMP_NUM_THREADS="1", OPENBLAS_NUM_THREADS="1", MKL_NUM_THREADS="1",
               VECLIB_MAXIMUM_THREADS="1", NUMEXPR_NUM_THREADS="1",
               PIP_CONFIG_FILE=os.devnull)
    mpl = Path(cache)/"matplotlib"
    mpl.mkdir(parents=True, exist_ok=True)
    env["MPLCONFIGDIR"] = str(mpl)
    return env


# This string is only the dependency test, not the ablation implementation.
_DEPENDENCY_PROBE = r'''
import io, json, platform, sys, traceback, importlib.metadata
try:
    import numpy as np
    import pandas as pd
    import scipy
    require_parquet = json.loads(sys.argv[2])
    if require_parquet:
        import pyarrow
    import matplotlib
    matplotlib.use("Agg", force=True)
    import matplotlib.pyplot as plt
    from matplotlib.backends.backend_pdf import PdfPages
    from sklearn.cross_decomposition import PLSRegression
    from sklearn.decomposition import PCA
    from sklearn.exceptions import ConvergenceWarning
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import (accuracy_score, average_precision_score,
        balanced_accuracy_score, confusion_matrix, f1_score, log_loss,
        matthews_corrcoef, roc_auc_score)
    from sklearn.model_selection import StratifiedGroupKFold
    from sklearn.preprocessing import StandardScaler
    from threadpoolctl import threadpool_limits
    with threadpool_limits(limits=1):
        rng = np.random.RandomState(13)
        X = rng.randn(48, 24)
        y = np.tile([0, 1], 24)
        X[:, :3] += y[:, None]
        Z = PLSRegression(n_components=16, scale=False).fit(X, y).transform(X)
        assert Z.shape == (48, 16) and np.isfinite(Z).all()
        assert PCA(n_components=16, svd_solver="full").fit_transform(X).shape == (48, 16)
        F = StandardScaler().fit_transform(Z)
        h = LogisticRegression(class_weight="balanced", max_iter=4000).fit(F, y)
        p = h.predict_proba(F)[:, 1]
        yp = (p >= .5).astype(int)
        assert np.isfinite([accuracy_score(y, yp), balanced_accuracy_score(y, yp),
            f1_score(y, yp), matthews_corrcoef(y, yp), roc_auc_score(y, p),
            average_precision_score(y, p), log_loss(y, p)]).all()
        assert confusion_matrix(y, yp).shape == (2, 2)
        groups = np.repeat(np.arange(12), 4)
        cv = StratifiedGroupKFold(3, shuffle=True, random_state=42)
        for a, b in cv.split(X, y, groups):
            assert not (set(groups[a]) & set(groups[b]))
        np.linalg.svd(X, full_matrices=False)
        if require_parquet:
            buf = io.BytesIO()
            frame = pd.DataFrame(X[:4, :3])
            frame.to_parquet(buf, engine="pyarrow", index=False)
            buf.seek(0)
            assert np.allclose(pd.read_parquet(buf, engine="pyarrow"), frame)
        fig, ax = plt.subplots(figsize=(2, 1))
        ax.plot([0, 1], [0, 1])
        for fmt in ("pdf", "svg", "png"):
            b = io.BytesIO()
            fig.savefig(b, format=fmt, dpi=300)
            assert len(b.getvalue()) > 100
        plt.close(fig)
    names = json.loads(sys.argv[1])
    versions = {name: importlib.metadata.version(name) for name in names}
    answer = {"ok": True, "versions": versions, "python": sys.version,
        "python_executable": sys.executable, "prefix": sys.prefix,
        "python_version": list(sys.version_info[:3]), "parquet_checked": require_parquet, "platform": platform.platform(),
        "machine": platform.machine(), "numpy_path": np.__file__}
except Exception as exc:
    answer = {"ok": False, "error_type": type(exc).__name__, "error": str(exc),
        "traceback": traceback.format_exc(), "python": sys.version,
        "python_version": list(sys.version_info[:3]), "machine": platform.machine()}
print("TQT_PROBE_JSON="+json.dumps(answer))
sys.exit(0 if answer["ok"] else 1)
'''


def _probe_environment(python_executable, runtime, cache):
    names = [r.split("==")[0] for r in PINNED_REQUIREMENTS]
    require_parquet = runtime.get("_require_parquet", True)
    if not require_parquet:
        names.remove("pyarrow")  # Synthetic smoke tests never open study Parquet.
    command = [str(python_executable), "-I", "-u", "-c", _DEPENDENCY_PROBE,
               json.dumps(names), json.dumps(require_parquet)]
    try:
        completed = subprocess.run(command, stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace",
            timeout=float(runtime["probe_timeout"]), env=_process_env(cache))
    except (OSError, subprocess.TimeoutExpired) as exc:
        return {"ok": False, "error_type": type(exc).__name__, "error": str(exc)}
    for line in reversed(completed.stdout.splitlines()):
        if line.startswith("TQT_PROBE_JSON="):
            value = json.loads(line.split("=", 1)[1])
            value["returncode"] = completed.returncode
            value["ok"] = bool(value["ok"] and completed.returncode == 0)
            return value
    return {"ok": False, "error_type": "ProbeProcessError",
            "error": "\n".join(completed.stdout.splitlines()[-20:]),
            "returncode": completed.returncode}


def _stream_process(command, log_path, env, cwd=None):
    """Blocking execution with visible progress and interrupt forwarding."""
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    options = {"start_new_session": True} if os.name != "nt" else {}
    tail = deque(maxlen=30)
    with log_path.open("a", encoding="utf-8") as log:
        log.write("\nCOMMAND: "+repr([str(x) for x in command])+"\n")
        log.flush()
        process = subprocess.Popen([str(x) for x in command], cwd=cwd, env=env,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
            encoding="utf-8", errors="replace", bufsize=1, **options)
        try:
            assert process.stdout is not None
            for line in process.stdout:
                log.write(line)
                log.flush()
                tail.append(line.rstrip())
                print(line, end="", flush=True)
            code = process.wait()
        except BaseException:
            if process.poll() is None:
                try:
                    if os.name != "nt":
                        os.killpg(process.pid, signal.SIGINT)
                    else:
                        process.terminate()
                    process.wait(timeout=8)
                except (OSError, subprocess.TimeoutExpired):
                    if process.poll() is None:
                        if os.name != "nt":
                            os.killpg(process.pid, signal.SIGKILL)
                        else:
                            process.kill()
                        process.wait()
            raise
        finally:
            if process.stdout is not None:
                process.stdout.close()
    if code:
        raise RuntimeError(f"Process exited with status {code}. Log: {log_path}\n"
                           +"\n".join(tail))


def _select_python(runtime, cache, launch_dir):
    # Do not resolve a venv's python symlink: that would discard venv isolation.
    base = str(Path(runtime["python_executable"] or sys.executable).expanduser().absolute())
    native = _probe_environment(base, runtime, cache)
    _save_json(launch_dir/"current_environment_probe.json", native)
    choice = runtime["environment"]
    if native["ok"] and choice in ("auto", "current"):
        print("Dependency check passed in a fresh process; using the existing Python installation.", flush=True)
        return base, {"selection": "current_fresh_process", "probe": native}
    if choice == "current":
        raise RuntimeError("The current environment failed its dependency check. "
            "Set RUNTIME['environment']='auto' or 'isolated' to recover without changing Anaconda.\n"
            +native.get("error", "Unknown dependency error"))
    if not native["ok"]:
        print("Current environment failed its dependency check: "
              f"{native.get('error_type')}: {native.get('error', '')[:350]}", flush=True)
        print("Using a separate compatible environment; no packages in the notebook kernel will be changed.", flush=True)
    version = native.get("python_version")
    if not version:
        info = subprocess.run([base, "-I", "-c", "import sys,json;print(json.dumps(list(sys.version_info[:3])))"],
            capture_output=True, text=True, timeout=30, env=_process_env(cache))
        if info.returncode:
            raise RuntimeError("Cannot execute the configured Python interpreter: "+info.stderr)
        version = json.loads(info.stdout.strip())
    if not ((3, 10) <= tuple(version[:2]) < (3, 14)):
        raise RuntimeError("Managed recovery uses package pins for Python 3.10--3.13. "
            "Use an existing healthy environment with environment='current', or point "
            "RUNTIME['python_executable'] to a Python 3.12 interpreter.")
    identity = {"base": base, "version": version[:2],
                "machine": native.get("machine", platform.machine()),
                "platform": sys.platform, "requirements": PINNED_REQUIREMENTS}
    key = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()[:16]
    root = cache/("env_py"+str(version[0])+str(version[1])+"_"+key)
    binary = root/("Scripts/python.exe" if os.name == "nt" else "bin/python")
    marker = root/"TQT_MANAGED_ENVIRONMENT.json"
    expected = dict(r.split("==", 1) for r in PINNED_REQUIREMENTS)
    if not runtime.get("_require_parquet", True):
        expected.pop("pyarrow", None)
    if binary.exists() and marker.exists():
        checked = _probe_environment(binary, runtime, cache)
        if checked["ok"] and all(checked["versions"].get(k) == v for k, v in expected.items()):
            print(f"Reusing compatible environment: {root}", flush=True)
            _save_json(launch_dir/"managed_environment_probe.json", checked)
            return str(binary), {"selection": "managed_cached", "root": str(root), "probe": checked}
    if not runtime["allow_install"]:
        raise RuntimeError("No usable compatible environment was found and allow_install=False. "
            "Enable installation into the dedicated venv, or supply a healthy python_executable. "
            "For offline recovery, supply RUNTIME['wheelhouse'] with all pinned wheels.")
    if root.exists() and not marker.exists():
        raise RuntimeError(f"Refusing to change an unrecognized environment directory: {root}")
    root.mkdir(parents=True, exist_ok=True)
    _save_json(marker, identity)
    setup_log = launch_dir/"environment_setup.log"
    if not binary.exists():
        print(f"Creating isolated environment: {root}", flush=True)
        _stream_process([base, "-I", "-m", "venv", str(root)], setup_log, _process_env(cache))
    # Fail before pip if this is not an isolated venv (protect the base environment).
    cfg_file = root/"pyvenv.cfg"
    if not cfg_file.exists() or "include-system-site-packages = false" not in cfg_file.read_text().lower():
        raise RuntimeError("The generated environment is not isolated; refusing package installation.")
    install = [str(binary), "-I", "-m", "pip", "--isolated", "--disable-pip-version-check",
        "install", "--no-input", "--only-binary=:all:", "--upgrade", "--force-reinstall",
        "--progress-bar", "off", "--cache-dir", str(cache/"pip")]
    if runtime["wheelhouse"]:
        wheels = Path(runtime["wheelhouse"]).expanduser().absolute()
        if not wheels.is_dir():
            raise FileNotFoundError(f"Offline wheelhouse not found: {wheels}")
        install += ["--no-index", "--find-links", str(wheels)]
        print(f"Installing the pinned stack from local wheels: {wheels}", flush=True)
    else:
        install += ["--index-url", "https://pypi.org/simple"]
        print("Installing the pinned binary stack from PyPI into the isolated environment only.", flush=True)
    requirements_file = root/"requirements_tqt_r1m1.txt"
    requirements_file.write_text("\n".join(PINNED_REQUIREMENTS)+"\n", encoding="utf-8")
    install += ["-r", str(requirements_file)]
    try:
        _stream_process(install, setup_log, _process_env(cache))
        _stream_process([str(binary), "-I", "-m", "pip", "check"], setup_log, _process_env(cache))
    except RuntimeError as exc:
        raise RuntimeError("Compatible-environment setup failed; the Anaconda installation was not modified. "
            "Internet access to PyPI (or a complete offline wheelhouse), writable disk space, and "
            f"compatible binary wheels are required. Details: {setup_log}\n{exc}") from exc
    checked = _probe_environment(binary, runtime, cache)
    _save_json(launch_dir/"managed_environment_probe.json", checked)
    if not checked["ok"] or not all(checked.get("versions", {}).get(k) == v for k, v in expected.items()):
        raise RuntimeError("The isolated environment did not pass its import/fit/Parquet/plot checks. "
            f"See {launch_dir/'managed_environment_probe.json'}. No analysis was run.")
    print("Compatible environment passed every dependency and rendering check.", flush=True)
    return str(binary), {"selection": "managed_created", "root": str(root), "probe": checked}


def _get_worker_source():
    """Obtain this file's visible worker code; supports %run and normal Jupyter cells."""
    try:
        return textwrap.dedent(inspect.getsource(_analysis_worker))
    except (OSError, TypeError):
        pass
    candidates = []
    source_file = globals().get("__file__")
    if source_file and Path(source_file).is_file():
        candidates.append(Path(source_file).read_text(encoding="utf-8"))
    try:
        ip = get_ipython()  # Provided by IPython; no scientific import in the parent.
        candidates.extend(reversed(ip.history_manager.input_hist_raw))
    except (NameError, AttributeError):
        pass
    for source in candidates:
        if "def _analysis_worker(" not in source:
            continue
        try:
            tree = ast.parse(source)
        except SyntaxError:
            continue
        for node in tree.body:
            if isinstance(node, ast.FunctionDef) and node.name == "_analysis_worker":
                return textwrap.dedent(ast.get_source_segment(source, node))+"\n"
    raise RuntimeError("Cannot locate the worker function source in this execution context. "
        "Run the saved standalone file with '%run R1_M1_TQT_ablation_robust.py', or import "
        "that file and call run(). Normal single-cell Jupyter execution is also supported.")


_WORKER_DRIVER = r'''
if __name__ == "__main__":
    import json, sys, traceback
    from pathlib import Path
    request_path, receipt_path = map(Path, sys.argv[1:3])
    request = json.loads(request_path.read_text(encoding="utf-8"))
    try:
        result = _analysis_worker(request["config"])
        out = Path(result["output_dir"]).absolute()
        execution = dict(request["execution"], worker_python=sys.executable,
                         worker_code_sha256=_R1M1_ENGINE_SHA256)
        (out/"execution_environment.json").write_text(json.dumps(execution, indent=2), encoding="utf-8")
        if result.get("status") in ("diagnostic_only", "preflight_only"):
            receipt = {"completed": False, "status": result["status"],
                "reason": result.get("reason", ""), "output_dir": str(out),
                "report_pdf": str(result["report_pdf"]),
                "config": result["config"], "data_audit": result["data_audit"],
                "execution": execution,
                "artifacts": sorted(str(p) for p in out.iterdir() if p.is_file())}
        else:
            receipt = {"completed": True, "status": "completed", "output_dir": str(out),
                "report_pdf": str(out/"R1_M1_ablation_report.pdf"),
                "figure_pdf": str(out/"R1_M1_ablation_figure.pdf"),
                "figure_svg": str(out/"R1_M1_ablation_figure.svg"),
                "figure_png": str(out/"R1_M1_ablation_figure.png"),
                "summary_csv": str(out/"summary.csv"),
                "summary": json.loads(result["summary"].to_json(orient="records", double_precision=15)),
                "self_tests": result["self_tests"], "audit": result["audit"], "data_audit": result["data_audit"],
                "config": result["config"], "execution": execution,
                "artifacts": sorted(str(p) for p in out.iterdir() if p.is_file())}
        receipt_path.write_text(json.dumps(receipt, indent=2, allow_nan=False), encoding="utf-8")
    except BaseException as exc:
        receipt_path.write_text(json.dumps({"completed": False,
            "error_type": type(exc).__name__, "error": str(exc),
            "traceback": traceback.format_exc()}, indent=2), encoding="utf-8")
        traceback.print_exc()
        sys.exit(1)
'''


def run(user_config=None, runtime=None):
    """Run all ablations synchronously; return paths and JSON-compatible results.

    The returned summary is a list of row dictionaries, not a pandas DataFrame;
    this deliberately avoids importing pandas/NumPy/sklearn in a broken kernel.
    Complete tabular results are also saved as CSV files in output_dir.
    """
    cfg, rt = copy.deepcopy(CONFIG), copy.deepcopy(RUNTIME)
    for overrides, target, label in ((user_config, cfg, "configuration"),
                                     (runtime, rt, "runtime")):
        if overrides is not None:
            if not isinstance(overrides, dict):
                raise TypeError(f"{label} overrides must be a dictionary.")
            unknown = set(overrides)-set(target)
            if unknown:
                raise ValueError(f"Unknown {label} keys: {sorted(unknown)}")
            target.update(copy.deepcopy(overrides))
    if rt["environment"] not in ("auto", "isolated", "current"):
        raise ValueError("environment must be 'auto', 'isolated', or 'current'.")
    if cfg["mode"] not in ("study", "smoke"):
        raise ValueError("mode must be 'study' or 'smoke'.")
    if float(rt["probe_timeout"]) <= 0:
        raise ValueError("probe_timeout must be positive.")
    for key in ("data_dir", "output_dir", "notebook_path", "groups_csv"):
        if cfg[key] is not None:
            cfg[key] = str(Path(cfg[key]).expanduser().absolute())
    if cfg["mode"] == "study":
        root = Path(cfg["data_dir"])
        if not root.is_dir():
            raise FileNotFoundError(f"Data directory does not exist: {root}")
        if not any(p.is_file() and not p.name.startswith(".")
                   and p.suffix.casefold() == ".parquet" for p in root.iterdir()):
            raise FileNotFoundError(f"No Parquet tensors found in {root}. "
                "The folder must contain the original sensor tensors. "
                "No synthetic results will be substituted.")
        print(f"Data directory: {root}", flush=True)
        print("Inputs: Parquet tensors only; PGM previews ignored. Duplicate-label records will be retained and grouped, not rejected.", flush=True)
    for key in ("groups_csv", "notebook_path"):
        if cfg[key] is not None and not Path(cfg[key]).is_file():
            raise FileNotFoundError(f"Configured {key} does not exist: {cfg[key]}")
    source = _get_worker_source()
    # Canonical AST identity is independent of temporary filenames, line numbers,
    # cell execution count and whitespace. Scientific changes invalidate caches.
    engine_hash = hashlib.sha256(ast.dump(ast.parse(source), include_attributes=False).encode()).hexdigest()
    cache = Path(rt["cache_dir"]).expanduser().absolute()
    cache.mkdir(parents=True, exist_ok=True)
    base_out = Path(str(cfg["output_dir"])+("_SMOKE_TEST" if cfg["mode"] == "smoke" else ""))
    launch_dir = base_out/"_launches"/(time.strftime("%Y%m%d_%H%M%S")+"_"+uuid.uuid4().hex[:8])
    launch_dir.mkdir(parents=True, exist_ok=True)
    rt["_require_parquet"] = cfg["mode"] == "study"
    python_executable, execution = _select_python(rt, cache, launch_dir)
    workers = cache/"workers"
    workers.mkdir(exist_ok=True)
    worker = workers/("r1m1_worker_"+engine_hash[:20]+".py")
    worker_text = ("from __future__ import annotations\n"
                   +"_R1M1_ENGINE_SHA256 = "+repr(engine_hash)+"\n\n"
                   +source+"\n"+_WORKER_DRIVER)
    # The generated source is derived only from this file, not remote notebooks.
    if not worker.exists() or worker.read_text(encoding="utf-8") != worker_text:
        temp = worker.with_name(worker.name+"."+uuid.uuid4().hex+".tmp")
        temp.write_text(worker_text, encoding="utf-8")
        temp.replace(worker)
    requested_display = bool(cfg["show_figures"])
    cfg["show_figures"] = False  # Rendering happens headlessly in the child.
    execution.update(launcher_python=sys.executable, launch_dir=str(launch_dir),
        requirements=list(PINNED_REQUIREMENTS) if execution["selection"].startswith("managed") else None)
    request, receipt = launch_dir/"request.json", launch_dir/"result.json"
    _save_json(request, {"config": cfg, "execution": execution})
    print(f"Running complete ablation implementation with: {python_executable}", flush=True)
    print(f"Execution log: {launch_dir/'analysis.log'}", flush=True)
    _stream_process([python_executable, "-I", "-u", str(worker), str(request), str(receipt)],
                    launch_dir/"analysis.log", _process_env(cache), cwd=str(Path.cwd()))
    if not receipt.exists():
        raise RuntimeError(f"Worker exited without a completion receipt; inspect {launch_dir/'analysis.log'}.")
    result = json.loads(receipt.read_text(encoding="utf-8"))
    if result.get("status") in ("diagnostic_only", "preflight_only"):
        print("\n" + ("Data audit completed; model evaluation was NOT performed."
              if result["status"] == "diagnostic_only" else "Preflight completed; model training was not requested."), flush=True)
        print(result.get("reason", ""), flush=True)
        print("Data-quality PDF: " + result["report_pdf"], flush=True)
        print("Audit files: " + result["output_dir"], flush=True)
        return result
    if not result.get("completed"):
        raise RuntimeError(result.get("error", "Ablation did not complete."))
    print("\nSVG manuscript figure: "+result["figure_svg"], flush=True)
    print("PDF manuscript figure: "+result["figure_pdf"], flush=True)
    print("300-dpi PNG figure: "+result["figure_png"], flush=True)
    print("Full PDF report: "+result["report_pdf"], flush=True)
    if requested_display:
        try:
            from IPython.display import SVG, display
            display(SVG(filename=result["figure_svg"]))
        except ImportError:
            print("Inline display needs IPython; the SVG/PDF files have already been saved.")
    return result


# ---------------------------------------------------------------------------
# COMPLETE SCIENTIFIC IMPLEMENTATION. Lazy by design: no scientific imports
# happen until this function is called INSIDE the checked worker process.
# ---------------------------------------------------------------------------

def _analysis_worker(_config):
    import ast
    import copy
    import hashlib
    import importlib.metadata
    import json
    import math
    import os
    import platform
    import random
    import re
    import sys
    import textwrap
    import time
    import warnings
    from collections import Counter, OrderedDict
    from pathlib import Path
    from typing import Any

    import numpy as np
    import pandas as pd
    import matplotlib
    matplotlib.use("Agg", force=True)
    import matplotlib.pyplot as plt
    from matplotlib.backends.backend_pdf import PdfPages
    from sklearn.cross_decomposition import PLSRegression
    from sklearn.decomposition import PCA
    from sklearn.exceptions import ConvergenceWarning
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import (accuracy_score, average_precision_score,
        balanced_accuracy_score, confusion_matrix, f1_score, log_loss,
        matthews_corrcoef, roc_auc_score)
    from sklearn.model_selection import StratifiedGroupKFold
    from sklearn.preprocessing import StandardScaler
    from threadpoolctl import threadpool_limits

    # ---------------------------------------------------------------------------
    # EDIT ONLY THIS BLOCK for normal use.
    # ---------------------------------------------------------------------------
    CONFIG = copy.deepcopy(_config)

    VERSION = "R1M1-1.3-DUPLICATE-SAFE"
    METRICS = ("accuracy", "balanced_accuracy", "f1", "mcc", "roc_auc", "average_precision")
    COUNT_METRICS = ("accuracy", "balanced_accuracy", "f1", "mcc")
    LABELS = {
        "majority": "Training-majority reference",
        "pls_lr": "PLS(16) + logistic regression",
        "pca_lr": "PCA(16) + logistic regression",
        "tanh_lr": "PLS + fixed tanh pooling + LR",
        "tqt_fixed": "PLS + fixed TQT-NB + LR",
        "tqt_random": "PLS + random frozen TQT-NB + LR",
        "tqt_trained": "PLS + trained TQT-NB + LR",
        "qt_exact": "PLS + exact notebook motifs + LR",
        "qt_exact_grid": "PLS + exact grid-matched motifs + LR",
        "pca_tqt": "PCA + trained TQT-NB + LR",
        "logical_tqt": "PLS + repaired logical TQT + LR",
        "logical_qt": "PLS + logical-CRY QT + LR",
    }


    def json_default(obj):
        if isinstance(obj, (np.integer, np.floating, np.bool_)):
            return obj.item()
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        if isinstance(obj, Path):
            return str(obj)
        raise TypeError(f"Cannot JSON-encode {type(obj)}")


    def write_json(path, obj):
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        temp = path.with_suffix(path.suffix + ".tmp")
        temp.write_text(json.dumps(obj, indent=2, default=json_default, allow_nan=False), encoding="utf-8")
        temp.replace(path)


    def digest(obj):
        return hashlib.sha256(json.dumps(obj, sort_keys=True, default=json_default).encode()).hexdigest()


    def stable_seed(*parts):
        return int(digest(parts)[:8], 16)


    def source_namespace():
        """Only an embedded, allowlisted numerical core; never runs notebook cells."""
        ns = {"np": np, "math": math, "random": random, "OrderedDict": OrderedDict}
        exec(compile(NOTEBOOK_CORE, "<embedded_current_notebook_core>", "exec"), ns)
        ns.update(GLOBAL_GATE_CACHE=ns["GateCache"](), DICT_BY_EPS={},
                  I2=np.eye(2, dtype=complex), CZ=np.diag([1, 1, 1, -1]).astype(complex))
        return ns


    # Verbatim function/class definitions from cell 0 of the CURRENT TQT notebook.
    # No old-paper model, dataset loader, training driver, plot, or output is embedded.
    NOTEBOOK_CORE = r'''def fib_F():
    phi = (1 + 5**0.5)/2
    a = 1.0/phi
    b = 1.0/(phi**0.5)
    return np.array([[a, b],[b,-a]], dtype=np.complex128)

def fib_R():
    R1 = np.exp(-4j*np.pi/5)
    Rtau = np.exp(3j*np.pi/5)
    return np.array([[R1,0],[0,Rtau]], dtype=np.complex128)

class FibQubit:
    def __init__(self):
        self.F = fib_F()
        self.R = fib_R()
        self.s1 = self.R.copy()
        self.s2 = self.F @ self.R @ self.F
        self.s1i = self.s1.conj().T
        self.s2i = self.s2.conj().T
    def apply_word(self, word):
        U = np.eye(2, dtype=np.complex128)
        for t in word:
            if t=='s1':   U = self.s1 @ U
            elif t=='s1-':U = self.s1i @ U
            elif t=='s2': U = self.s2 @ U
            elif t=='s2-':U = self.s2i @ U
            else: raise ValueError(t)
        return U

def Rx(theta):
    c = math.cos(theta/2); s = math.sin(theta/2)
    return np.array([[c, -1j*s],[-1j*s, c]], dtype=np.complex128)

def Rz(theta):
    return np.array([[np.exp(-1j*theta/2), 0],[0, np.exp(1j*theta/2)]], dtype=np.complex128)

def Ry(theta):
    c = math.cos(theta/2); s = math.sin(theta/2)
    return np.array([[c, -s],[s, c]], dtype=np.complex128)

def unitary_distance(U,V):
    t = np.trace(U.conj().T @ V)
    phase = 1.0+0j if abs(t)<1e-12 else t/abs(t)
    return np.linalg.norm(U - phase*V, ord='fro')/2

class BraidStats:
    def __init__(self):
        self.lengths = []
        self.errors  = []
        self.tokens  = []
        self.bigrams = {}
    def add(self, word, err):
        self.lengths.append(len(word))
        self.errors.append(float(err))
        self.tokens.extend(word)
        for a,b in zip(word[:-1], word[1:]):
            self.bigrams[(a,b)] = self.bigrams.get((a,b), 0) + 1
    def merge(self, other):
        self.lengths += other.lengths
        self.errors  += other.errors
        self.tokens  += other.tokens
        for k,v in other.bigrams.items():
            self.bigrams[k] = self.bigrams.get(k,0)+v
    def summary(self):
        if not self.lengths:
            return {"avg_len":0.0,"avg_err":0.0,"n":0}
        return {"avg_len":float(np.mean(self.lengths)),
                "avg_err":float(np.mean(self.errors)),
                "n":len(self.lengths)}

class GateCache:
    def __init__(self, maxsize=80000, quant=np.pi/128):
        self.store = OrderedDict()
        self.maxsize = maxsize
        self.quant = float(quant)
    def key(self, axis, theta, eps):
        if axis is None or theta is None: return None
        tq = float(np.round(theta / self.quant) * self.quant)
        return (axis, tq, int(round(eps)))
    def get(self, axis, theta, eps):
        k = self.key(axis, theta, eps)
        if k is None: return None
        if k in self.store:
            v = self.store.pop(k); self.store[k]=v
            return v
        return None
    def put(self, axis, theta, eps, value):
        k = self.key(axis, theta, eps)
        if k is None: return
        if k in self.store: self.store.pop(k)
        self.store[k]=value
        if len(self.store) > self.maxsize:
            self.store.popitem(last=False)

def unitary_key(U, bins=2048):
    t = np.trace(U)
    phase = 1.0+0j if abs(t)<1e-12 else t/abs(t)
    W = U / phase
    r = np.array([W[0,0], W[0,1], W[1,0], W[1,1]])
    q = np.round(np.concatenate([r.real, r.imag]) * bins).astype(np.int64)
    return tuple(q.tolist())

def build_dict(fib, L0=8):
    gens = [['s1'],['s1-'],['s2'],['s2-']]
    words = [()]
    table = {}
    for _ in range(L0):
        new_words = []
        for w in words:
            U = fib.apply_word(list(w)) if w else np.eye(2, dtype=np.complex128)
            key = unitary_key(U)
            if key not in table:
                table[key] = (list(w), U)
            for g in gens:
                new_words.append(tuple(list(w)+g))
        words = new_words
    return table

def mitm_compile(Ut, fib, D, stats=None):
    keys = list(D.keys())
    if not keys: return None
    best = None
    for _ in range(1024):
        wA, UA = D[random.choice(keys)]
        wB, UB = D[random.choice(keys)]
        U = (fib.apply_word(wA) if wA else np.eye(2)) @ (fib.apply_word(wB) if wB else np.eye(2))
        d = unitary_distance(U, Ut)
        if (best is None) or (d < best[0]):
            best = (d, wA, wB, U)
            if d < 1e-3: break
    if best is None: return None
    d, wA, wB, U = best
    word = (wA or []) + (wB or [])
    if stats: stats.add(word, d)
    return word, U, d

def compile_to_braid(Ut, fib, epsilon=10, stats: BraidStats=None, axis_hint=None, theta_hint=None):
    theta_q = None if theta_hint is None else float(np.round(theta_hint / GLOBAL_GATE_CACHE.quant) * GLOBAL_GATE_CACHE.quant)
    cached = GLOBAL_GATE_CACHE.get(axis_hint, theta_q, epsilon)
    if cached is not None:
        word, U, err = cached
        if stats: stats.add(word, err)
        return word, U, err

    eps_key = int(round(epsilon))
    if eps_key not in DICT_BY_EPS:
        DICT_BY_EPS[eps_key] = build_dict(FibQubit(), L0=8)
    cand = mitm_compile(Ut, FibQubit(), DICT_BY_EPS[eps_key], stats=None)
    if cand is not None:
        word, U, d = cand
        if stats: stats.add(word, d)
        GLOBAL_GATE_CACHE.put(axis_hint, theta_q, epsilon, (word, U, d))
        return word, U, d

    tol = 10 ** (-(epsilon - 6)/2.0)
    max_len = max(0, int(round(6 + 4*(epsilon - 8))))
    gens = [['s1'],['s1-'],['s2'],['s2-']]
    fib = FibQubit()
    word=[]; U=np.eye(2, dtype=np.complex128)
    best = unitary_distance(U, Ut)
    if best<tol:
        if stats: stats.add(word, best)
        GLOBAL_GATE_CACHE.put(axis_hint, theta_q, epsilon, (word, U, best))
        return word, U, best
    for _ in range(max_len):
        bestd=None; bestg=None; bestU=None
        for g in gens:
            Utrial = fib.apply_word(word+g)
            d = unitary_distance(Utrial, Ut)
            if bestd is None or d<bestd: bestd, bestg, bestU = d, g, Utrial
        word += bestg; U=bestU; best=bestd
        if best<tol: break
    if stats: stats.add(word, best)
    GLOBAL_GATE_CACHE.put(axis_hint, theta_q, epsilon, (word, U, best))
    return word, U, best

def kron(A,B): return np.kron(A,B)

def apply_1q(U4, U2, which):
    G = kron(U2, I2) if which==0 else kron(I2, U2)
    return G @ U4

def apply_CZ(U4, epsilon, stats: BraidStats):
    tol = 10 ** (-(epsilon - 6)/2.0)
    L = max(1, int(round(16 + 6*(epsilon - 8))))
    word = ['E'] * L
    stats.add(word, tol/2.0)
    return CZ @ U4

def compile_and_apply_1q(U4, Ut, which, epsilon, stats, axis_hint, theta_hint):
    _, Uapp, _ = compile_to_braid(Ut, FibQubit(), epsilon=epsilon, stats=stats,
                                  axis_hint=axis_hint, theta_hint=theta_hint)
    return apply_1q(U4, Uapp, which)

def apply_H(U4, which, epsilon, stats):
    for ax, th in [('z', np.pi), ('x', np.pi/2), ('z', np.pi)]:
        Ut = Rz(th) if ax=='z' else Rx(th)
        U4 = compile_and_apply_1q(U4, Ut, which, epsilon, stats, axis_hint=ax, theta_hint=th)
    return U4

def apply_CNOT(U4, ctrl, tgt, epsilon, stats):
    U4 = apply_H(U4, tgt, epsilon, stats)
    U4 = apply_CZ(U4, epsilon, stats)
    U4 = apply_H(U4, tgt, epsilon, stats)
    return U4

def apply_CRZ(U4, theta, ctrl, tgt, epsilon, stats):
    U4 = apply_CNOT(U4, ctrl, tgt, epsilon, stats)
    U4 = compile_and_apply_1q(U4, Rz(theta), tgt, epsilon, stats, axis_hint='z', theta_hint=theta)
    U4 = apply_CNOT(U4, ctrl, tgt, epsilon, stats)
    return U4

def apply_CRX(U4, theta, ctrl, tgt, epsilon, stats):
    U4 = apply_H(U4, tgt, epsilon, stats)
    U4 = apply_CRZ(U4, theta, ctrl, tgt, epsilon, stats)
    U4 = apply_H(U4, tgt, epsilon, stats)
    return U4

def apply_CRY(U4, theta, ctrl, tgt, epsilon, stats):
    U4 = compile_and_apply_1q(U4, Rz(-np.pi/2), tgt, epsilon, stats, axis_hint='z', theta_hint=-np.pi/2)
    U4 = apply_CRX(U4, theta, ctrl, tgt, epsilon, stats)
    U4 = compile_and_apply_1q(U4, Rz(np.pi/2),  tgt, epsilon, stats, axis_hint='z', theta_hint=np.pi/2)
    return U4

def expect_Z_on_target(psi, tgt):
    a,b,c,d = psi
    if tgt==1:
        return float((abs(a)**2 + abs(c)**2) - (abs(b)**2 + abs(d)**2))
    else:
        return float((abs(a)**2 + abs(b)**2) - (abs(c)**2 + abs(d)**2))

def measure_triplet(U4, tgt, epsilon, stats):
    psi = U4 @ np.array([1,0,0,0], dtype=np.complex128)
    z = expect_Z_on_target(psi, tgt)
    Ux = compile_and_apply_1q(U4, Ry(-np.pi/2), tgt, epsilon, stats, axis_hint='y', theta_hint=-np.pi/2)
    zx = expect_Z_on_target(Ux @ np.array([1,0,0,0], dtype=np.complex128), tgt)
    Uy = compile_and_apply_1q(U4, Rx(np.pi/2),  tgt, epsilon, stats, axis_hint='x', theta_hint=np.pi/2)
    zy = expect_Z_on_target(Uy @ np.array([1,0,0,0], dtype=np.complex128), tgt)
    return np.array([z, zx, zy], dtype=np.float64)

class CodebookAnyonTransistor:
    """
    Three-stage anyonic transistor using codebook rotations + real CRY(π/3).
    Parameters to *learn*: alpha = (α1, α2, α3) (global signal-to-angle scales per stage).
    Features: concatenated Pauli triplets from Stage 1 (8 blocks→24d), Stage 2 (6→18d), Stage 3 (4→12d) → 54-dim.
    """
    def __init__(self, epsilon=12, seed=0,
                 codebook=(0.5, 1.0, 1.5, 2.0)):
        self.epsilon = epsilon
        self.code = list(codebook)
        self.stats = BraidStats()
        rs = np.random.RandomState(seed)
        self.R1 = np.linalg.qr(rs.randn(16, 8))[0]     # 16x8
        self.R2 = np.linalg.qr(rs.randn(16, 6))[0]     # 16x6
        self.R3 = np.linalg.qr(rs.randn(16, 4))[0]     # 16x4

    def _block(self, s, alpha_stage):
        eps = self.epsilon
        U4 = np.eye(4, dtype=np.complex128)
        U4 = compile_and_apply_1q(U4, Rx(np.pi), which=0, epsilon=eps, stats=self.stats,
                                  axis_hint='x', theta_hint=np.pi)
        for c in self.code:
            th = float(alpha_stage * s * c * np.pi)
            U4 = compile_and_apply_1q(U4, Rz(th), which=1, epsilon=eps, stats=self.stats,
                                      axis_hint='z', theta_hint=th)
            U4 = compile_and_apply_1q(U4, Rx(th), which=1, epsilon=eps, stats=self.stats,
                                      axis_hint='x', theta_hint=th)
        U4 = apply_CRY(U4, theta=np.pi/3, ctrl=0, tgt=1, epsilon=eps, stats=self.stats)
        return measure_triplet(U4, tgt=1, epsilon=eps, stats=self.stats)

    def features(self, Z16, alpha):
        N = Z16.shape[0]
        out = np.zeros((N, 54), dtype=np.float64)
        s1 = np.tanh(Z16 @ self.R1)   # (N,8)
        s2 = np.tanh(Z16 @ self.R2)   # (N,6)
        s3 = np.tanh(Z16 @ self.R3)   # (N,4)

        idx = 0
        for n in range(N):
            vec = [self._block(s1[n,i], alpha[0]) for i in range(8)]
            out[n, idx:idx+24] = np.concatenate(vec, axis=0)
        idx += 24
        for n in range(N):
            vec = [self._block(s2[n,i], alpha[1]) for i in range(6)]
            out[n, idx:idx+18] = np.concatenate(vec, axis=0)
        idx += 18
        for n in range(N):
            vec = [self._block(s3[n,i], alpha[2]) for i in range(4)]
            out[n, idx:idx+12] = np.concatenate(vec, axis=0)
        return out'''

    # ---------------------------------------------------------------------------
    # Numerical audit, deterministic compilation, vectorized matched feature maps.
    # ---------------------------------------------------------------------------

    def phase_free_distance(U, V):
        """min_phi ||U - exp(i phi)V||_F/2 (same /2 convention as manuscript)."""
        t = np.vdot(U, V)
        phase = np.conjugate(t)/abs(t) if abs(t) > 1e-14 else 1.0
        return float(np.linalg.norm(U-phase*V, "fro")/2)


    def rotations(axis, theta):
        theta = np.asarray(theta, dtype=float).reshape(-1)
        c, s = np.cos(theta/2), np.sin(theta/2)
        U = np.zeros((len(theta), 2, 2), dtype=complex)
        if axis == "x":
            U[:, 0, 0] = U[:, 1, 1] = c
            U[:, 0, 1] = U[:, 1, 0] = -1j*s
        elif axis == "y":
            U[:, 0, 0] = U[:, 1, 1] = c
            U[:, 0, 1], U[:, 1, 0] = -s, s
        elif axis == "z":
            U[:, 0, 0], U[:, 1, 1] = np.exp(-1j*theta/2), np.exp(1j*theta/2)
        else:
            raise ValueError(f"Unknown axis: {axis}")
        return U


    def logical_cry(theta=np.pi/3):
        U = np.eye(4, dtype=complex)
        U[2:, 2:] = rotations("y", [theta])[0]
        return U


    def check_notebook_provenance(cfg):
        embedded = ast.parse(NOTEBOOK_CORE)
        names = {n.name for n in embedded.body if isinstance(n, (ast.FunctionDef, ast.ClassDef))}
        reference = ast.dump(embedded, include_attributes=False)
        paths = ([Path(cfg["notebook_path"])] if cfg["notebook_path"] else
                 sorted(Path.cwd().glob("Q_transistor_Topological_QC*.ipynb")))
        details = {"embedded_core_sha256": hashlib.sha256(NOTEBOOK_CORE.encode()).hexdigest(),
                   "notebooks_checked": [], "uses_earlier_paper_results": False}
        for p in paths:
            if not p.exists():
                raise FileNotFoundError(p)
            nb = json.loads(p.read_text(encoding="utf-8"))
            found = False
            for cell in nb.get("cells", []):
                src = "".join(cell.get("source", []))
                if cell.get("cell_type") != "code" or "class CodebookAnyonTransistor" not in src:
                    continue
                tree = ast.parse(src)
                chosen = [n for n in tree.body
                          if isinstance(n, (ast.FunctionDef, ast.ClassDef)) and n.name in names]
                actual = ast.dump(ast.Module(body=chosen, type_ignores=[]), include_attributes=False)
                if actual != reference:
                    raise RuntimeError(f"{p.name} has a different numerical core. This script is tied "
                        "to the supplied current TQT notebook; review the changes before using it.")
                found = True
                break
            if not found:
                raise RuntimeError(f"No matching current TQT class found in {p}.")
            details["notebooks_checked"].append({"name": p.name,
                "sha256": hashlib.sha256(p.read_bytes()).hexdigest()})
        if not paths:
            details["note"] = "Original notebook absent; using its embedded, provenance-recorded core."
        return details


    class FrozenCompiler:
        """Deterministic, label-independent LUT keyed by (mode, axis, angle bin).

        Notebook mode preserves the source's *matrix* and distance choices, including
        its defects. Canonical bin centers and per-key RNG remove first-caller/cache
        order effects. This is intentionally NOT a bit-for-bit historical rerun.

        Repaired mode uses the same dictionary and sampled candidates, but fixes the
        phase metric and emits a word whose replay actually equals the chosen matrix.
        epsilon is fixed at 12; no claim of an epsilon-dependent search budget is made.
        """
        def __init__(self, cfg):
            self.cfg = cfg
            self.quant = float(cfg["angle_quantum"])
            self.ns = source_namespace()
            self.fib = self.ns["FibQubit"]()
            table = self.ns["build_dict"](self.fib, L0=int(cfg["compiler_dictionary_depth"]))
            self.dictionary = table
            self.items = list(table.values())
            self.matrices = np.stack([v[1] for v in self.items])
            self.cache = {}

        def compile_key(self, mode, axis, qindex):
            key = (mode, axis, int(qindex))
            if key in self.cache:
                return self.cache[key]
            target = rotations(axis, [qindex*self.quant])[0]
            seed = stable_seed(self.cfg["compiler_seed"], self.cfg["epsilon"], axis, int(qindex))
            rng = random.Random(seed)
            k, n = int(self.cfg["compiler_trials"]), len(self.items)
            # Python Random.choice draws in the same order as the notebook, with an
            # independent RNG per key instead of the global, evaluation-order RNG.
            pairs = np.array([[rng.randrange(n), rng.randrange(n)] for _ in range(k)])
            U = self.matrices[pairs[:, 0]] @ self.matrices[pairs[:, 1]]
            t = np.einsum("nij,ij->n", U.conj(), target)
            ph = np.ones(k, dtype=complex)
            good = np.abs(t) >= 1e-12
            ph[good] = t[good]/np.abs(t[good])
            if mode == "repaired":
                ph = ph.conj()
            elif mode != "notebook":
                raise ValueError(mode)
            d = np.linalg.norm(U-ph[:, None, None]*target, axis=(1, 2))/2
            early = np.flatnonzero(d < 1e-3)
            pick = int(early[0] if early.size else np.argmin(d))
            ia, ib = pairs[pick]
            wa, wb = self.items[ia][0], self.items[ib][0]
            # Source word is wa+wb although its stored matrix is UA@UB. Applying a
            # word left-to-right to a state instead replays UB@UA.
            word = list(wb)+list(wa) if mode == "repaired" else list(wa)+list(wb)
            result = dict(mode=mode, axis=axis, qindex=int(qindex), theta=float(qindex*self.quant),
                word=word, U=U[pick], selection_distance=float(d[pick]),
                phase_free_distance=phase_free_distance(U[pick], target),
                word_replay_distance=phase_free_distance(self.fib.apply_word(word), U[pick]))
            self.cache[key] = result
            return result

        def gates(self, mode, axis, theta, counter=None):
            theta = np.asarray(theta, dtype=float).reshape(-1)
            if not np.isfinite(theta).all():
                raise FloatingPointError("Nonfinite rotation angles.")
            if mode in ("exact", "exact_grid"):
                if mode == "exact_grid":
                    theta = np.rint(theta/self.quant)*self.quant
                return rotations(axis, theta)
            bins, inverse, counts = np.unique(np.rint(theta/self.quant).astype(np.int64),
                                             return_inverse=True, return_counts=True)
            mats = []
            for b, count in zip(bins, counts):
                entry = self.compile_key(mode, axis, int(b))
                mats.append(entry["U"])
                if counter is not None:
                    counter[(mode, axis, int(b))] += int(count)
            return np.stack(mats)[inverse]

        def inventory(self, path):
            """Notebook words are diagnostic only; repaired words pass replay checks."""
            with Path(path).open("w", encoding="utf-8") as out:
                for key in sorted(self.cache):
                    row = dict(self.cache[key])
                    U = row.pop("U")
                    row.update(matrix_real=U.real.tolist(), matrix_imag=U.imag.tolist(),
                               executable_word_verified=bool(row["word_replay_distance"] < 1e-10))
                    out.write(json.dumps(row, default=json_default)+"\n")


    class FeatureEngine:
        """Exactly 18 parallel pooled signals and 54 post-bias readout features.

        Unlike the earlier paper, this is NOT a 4->3->2 cascaded QNN. R1/R2/R3 each
        act directly on the same 16-dimensional contraction, as in the current code.
        """
        def __init__(self, cfg, compiler, kind="notebook"):
            self.cfg, self.compiler, self.kind = cfg, compiler, kind
            self.mode = {"notebook": "notebook", "exact_nb": "exact",
                "exact_grid_nb": "exact_grid", "logical_tqt": "repaired",
                "logical_qt": "exact"}[kind]
            rs = np.random.RandomState(cfg["projection_seed"])
            self.projections = tuple(np.linalg.qr(rs.randn(16, n))[0] for n in (8, 6, 4))
            self.constant_counts = Counter()
            def one(axis, theta):
                return compiler.gates(self.mode, axis, [theta], self.constant_counts)[0]
            self.prep = one("x", np.pi)[:, 0]
            if kind in ("logical_tqt", "logical_qt"):
                self.bias = logical_cry()
                self.logical_entanglers = 1
            else:
                ns = source_namespace()
                def gate(Ut, fib, epsilon=12, stats=None, axis_hint=None, theta_hint=None):
                    return [], one(axis_hint, theta_hint), 0.0
                ns["compile_to_braid"] = gate
                self.bias = ns["apply_CRY"](np.eye(4, dtype=complex), np.pi/3, 0, 1,
                                             cfg["epsilon"], ns["BraidStats"]())
                self.logical_entanglers = 2  # Actual CZ matrices in the source function.
            self.basis_x = np.kron(np.eye(2), one("y", -np.pi/2))
            self.basis_y = np.kron(np.eye(2), one("x", np.pi/2))

        def pool(self, Z):
            Z = np.asarray(Z, dtype=float)
            if Z.ndim != 2 or Z.shape[1] != 16:
                raise ValueError("Current notebook architecture requires exactly 16 contraction signals.")
            return np.concatenate([np.tanh(Z@R) for R in self.projections], axis=1)

        def features(self, Z, alpha, collect=False):
            pooled = self.pool(Z)
            scales = np.repeat(np.asarray(alpha, dtype=float), (8, 6, 4))
            angle = (pooled*scales).reshape(-1)*np.pi
            count = Counter() if collect else None
            v = np.zeros((angle.size, 2), dtype=complex)
            v[:, 0] = 1
            for c in self.cfg["codebook"]:
                for axis in ("z", "x"):
                    U = self.compiler.gates(self.mode, axis, c*angle, count)
                    v = np.einsum("nij,nj->ni", U, v)
            psi = np.einsum("i,nj->nij", self.prep, v).reshape(-1, 4) @ self.bias.T
            sign = np.array([1, -1, 1, -1])
            z = (np.abs(psi)**2)@sign
            x = (np.abs(psi@self.basis_x.T)**2)@sign
            y = (np.abs(psi@self.basis_y.T)**2)@sign
            F = np.column_stack((z, x, y)).reshape(len(Z), 54)
            if not np.isfinite(F).all() or np.max(np.abs(F)) > 1+1e-8:
                raise FloatingPointError("Unphysical or nonfinite expectation features.")
            resources = {}
            if collect:
                for key, n in self.constant_counts.items():
                    count[key] += n*len(angle)
                total = sum(count.values())
                resources = dict(single_qubit_actions=total,
                    actual_logical_entanglers_per_sample=18*self.logical_entanglers,
                    compiler_mode=self.mode, sampled_shots=0,
                    average_word_length=None, average_phase_free_distance=None,
                    average_notebook_word_replay_distance=None)
                if total:
                    resources.update(average_word_length=sum(len(self.compiler.cache[k]["word"])*v
                        for k, v in count.items())/total,
                        average_phase_free_distance=sum(self.compiler.cache[k]["phase_free_distance"]*v
                        for k, v in count.items())/total,
                        average_notebook_word_replay_distance=sum(self.compiler.cache[k]["word_replay_distance"]*v
                        for k, v in count.items())/total)
            return F, resources


    def implementation_audit(cfg, compiler):
        ns = source_namespace()
        ns["compile_to_braid"] = lambda Ut, fib, **kw: ([], Ut, 0.0)
        U = ns["Rx"](.47)
        test_phase = ns["unitary_distance"](U, np.exp(.71j)*U)
        h = ns["apply_H"](np.eye(4), 1, 12, ns["BraidStats"]())
        true_h = np.kron(np.eye(2), np.array([[1, 1], [1, -1]])/np.sqrt(2))
        cry = ns["apply_CRY"](np.eye(4), np.pi/3, 0, 1, 12, ns["BraidStats"]())
        entry = compiler.compile_key("notebook", "x", 29)
        repaired = compiler.compile_key("repaired", "x", 29)
        measured = dict(original_distance_on_global_phase_equivalent_gates=float(test_phase),
            correct_distance_on_same_gates=phase_free_distance(U, np.exp(.71j)*U),
            exact_notebook_H_vs_H=phase_free_distance(h, true_h),
            exact_notebook_CRY_vs_logical_CRY=phase_free_distance(cry, logical_cry()),
            example_notebook_word_replay_distance=entry["word_replay_distance"],
            example_repaired_word_replay_distance=repaired["word_replay_distance"])
        notes = [
            "TQT-NB denotes the current notebook's numerical matrix implementation, not a certified physical braid realization.",
            "The source distance uses the non-conjugated phase. It is not global-phase invariant. The main notebook arm retains this selection objective; the repaired arm corrects it.",
            "The source concatenates wA+wB but returns UA@UB, while apply_word replays UB@UA. Notebook words are exported as diagnostics, never as verified executable programs.",
            "The source exact apply_H and apply_CRY compositions do not equal Hadamard and controlled-Ry. Main notebook/exact-motif arms keep those compositions. The separate logical pair uses an exact block-diagonal CRY(pi/3).",
            "The successful source compiler branch always builds L0=8 and samples 1024 pairs. epsilon does not change that budget. This ablation fixes epsilon=12 and makes no new epsilon-sweep claim.",
            "The source cache rounds keys but compiles the first unrounded target reaching a key. This code compiles canonical bin centers with independent per-key RNG: deterministic, immutable and label-independent. Historical scores are not reproduced bit-for-bit.",
            "All three stages pool the same 16 signals in parallel; the 54 post-bias features and gate order match the executable notebook. The manuscript's pre-bias/triplet descriptions are not silently substituted.",
            "Global input statistics and PLS are refitted within every inner-training split and on outer-training only for refit. No validation/test label fits a contraction used to select its own hyperparameters.",
            "The source filename prefix is not evidence of subject identity. Without a verified subject mapping, all outputs say filename-group-disjoint, not subject-disjoint.",
            "epsilon=12 was historically identified using reported test sweeps. New ablations condition on it; reusing the same cohort is not independent confirmation of its optimality.",
            "The earlier QT notebooks use image inputs, a learned linear contraction and cascaded 4/3/2 blocks. Their results and weights are excluded. The exact-motif comparator is architecture-matched to the current notebook, not a reproduction of the old model.",
            "No noise or finite-shot model is simulated; predictive gains alone do not establish quantum advantage, fault tolerance, or physical topological robustness.",
        ]
        assert measured["correct_distance_on_same_gates"] < 1e-12
        assert repaired["word_replay_distance"] < 1e-10
        assert np.allclose(logical_cry().conj().T@logical_cry(), np.eye(4))
        return {"numerical_checks": measured, "interpretation_notes": notes}


    def numerical_self_tests(cfg, compiler):
        """Check batch implementation against the source's scalar 4x4 calculation."""
        ns = source_namespace()
        def gate(Ut, fib, epsilon=12, stats=None, axis_hint=None, theta_hint=None):
            U = compiler.gates("notebook", axis_hint, [theta_hint])[0]
            return [], U, 0.0
        ns["compile_to_braid"] = gate
        original = ns["CodebookAnyonTransistor"](epsilon=cfg["epsilon"],
            seed=cfg["projection_seed"], codebook=cfg["codebook"])
        engine = FeatureEngine(cfg, compiler, "notebook")
        Z = np.random.RandomState(912).normal(size=(2, 16))*.2
        alpha = np.array([.6, .8, 1.1])
        slow = original.features(Z, alpha)
        fast, _ = engine.features(Z, alpha)
        error = float(np.max(np.abs(slow-fast)))
        if error > 1e-10:
            raise AssertionError(f"Vectorized/source features disagree: {error}")
        assert np.allclose(fast, engine.features(Z[::-1], alpha)[0][::-1], atol=1e-12)
        assert np.allclose(fast[:1], engine.features(Z[:1], alpha)[0], atol=1e-12)
        # Check vectorized compiler against original 1024-trial selection with the
        # same per-key RNG, not only against another call to the vectorized code.
        axis, qindex = "z", 19
        seed = stable_seed(cfg["compiler_seed"], cfg["epsilon"], axis, qindex)
        ref = source_namespace()
        ref["random"] = random.Random(seed)
        # Source hard-codes 1024 trials. Smoke may use a smaller budget, so the
        # original-compiler equivalence test is applicable only to the study budget.
        compile_error = None
        if cfg["compiler_trials"] == 1024:
            target = rotations(axis, [qindex*compiler.quant])[0]
            _, Uref, _ = ref["mitm_compile"](target, ref["FibQubit"](), compiler.dictionary)
            Ufast = compiler.compile_key("notebook", axis, qindex)["U"]
            compile_error = float(np.linalg.norm(Uref-Ufast))
            if compile_error > 1e-10:
                raise AssertionError("Vectorized compiler differs from supplied compiler.")
        return dict(maximum_batch_vs_source_feature_error=error,
                    original_vs_vectorized_compiler_error=compile_error,
                    row_order_and_batch_invariance=True)
    # ---------------------------------------------------------------------------
    # Data contract: one parquet tensor per record; no prefit PLS or leaked globals.
    # ---------------------------------------------------------------------------

    def parse_tensor_filename(filename):
        """Return (filename-group, binary label, source class) from a full suffix.

        Match explicit negative suffixes first: checking only whether a stem
        contains/ends in 'walking' would also match a negative class. Prefixes
        remain case-sensitive identifiers; only the known suffix is folded.
        """
        name = Path(filename).name
        lowered = name.casefold()
        suffixes = (
            ("-not_walking.parquet", 0, "not_walking"),
            ("-not-walking.parquet", 0, "not_walking"),
            ("-not walking.parquet", 0, "not_walking"),
            ("-standing.parquet", 0, "standing"),
            ("-walking.parquet", 1, "walking"),
        )
        for suffix, label, source_class in suffixes:
            if lowered.endswith(suffix):
                group = name[:-len(suffix)]
                if not group.strip():
                    raise ValueError(f"Missing record identifier before class suffix: {name}")
                return group, label, source_class
        raise ValueError(
            f"Unrecognized filename label: {name}. Expected '-walking.parquet' "
            "(class 1), '-not_walking.parquet' (class 0), or the legacy "
            "'-standing.parquet' (class 0). '-not-walking.parquet' and "
            "'-not walking.parquet' are also accepted as class 0. "
            "Unrelated filenames are not silently assigned a class."
        )


    def load_data(cfg):
        if cfg["mode"] == "smoke":
            rng = np.random.RandomState(8147)
            groups = np.repeat(np.arange(36), 4)
            y = np.tile([0, 0, 1, 1], 36)
            X = rng.normal(size=(len(y), 48))
            X[:, :5] += (.75*(2*y-1))[:, None]
            X[:, 5:10] += rng.normal(size=(36, 5))[groups]*.4
            arrays = [a.reshape(6, 8).copy() for a in X]
            # Regression fixture: one same-label and one mixed-label exact
            # identity across groups, plus distinct raw tensors sharing a prefix.
            arrays[5] = arrays[0].copy()
            arrays[6] = arrays[0].copy()   # y[0]=0, y[6]=1: retain both labels.
            arrays[11] = arrays[2].copy()
            arrays[33] = np.column_stack([arrays[34], np.ones(6)])
            names = [f"SYNTHETIC_segment_{i:04d}_tensor-{'walking' if v else 'not_walking'}.parquet"
                     for i, v in enumerate(y)]
            data = dict(arrays=arrays, y=y.astype(int), groups=groups.astype(str), names=names,
                group_description="synthetic-group-disjoint", synthetic=True,
                source_classes=["walking" if int(v) == 1 else "not_walking" for v in y],
                file_hashes=[hashlib.sha256(a.tobytes()).hexdigest() for a in arrays])
        else:
            root = Path(cfg["data_dir"]).expanduser()
            if not root.is_dir():
                raise FileNotFoundError(f"Data directory does not exist: {root}")
            # Skip hidden macOS metadata/AppleDouble files, never real tensors.
            paths = sorted(p for p in root.iterdir() if p.is_file()
                           and not p.name.startswith(".")
                           and p.suffix.casefold() == ".parquet")
            if not paths:
                raise FileNotFoundError(f"No Parquet tensors found in {root.resolve()}. "
                    "Set CONFIG['data_dir'] to the original outs directory. "
                    "No synthetic data will be silently substituted.")
            # Validate all names before loading large arrays or fitting any model.
            parsed_names = [parse_tensor_filename(p.name) for p in paths]
            arrays, y, groups, names, hashes, source_classes = [], [], [], [], [], []
            for p, (group, label, source_class) in zip(paths, parsed_names):
                try:
                    arr = pd.read_parquet(p).to_numpy(dtype=np.float64)
                except ImportError as exc:
                    raise ImportError("Parquet loading failed despite dependency preflight; inspect the worker execution log.") from exc
                except (TypeError, ValueError) as exc:
                    raise ValueError(f"{p.name} must contain only numeric sensor-tensor columns.") from exc
                if arr.ndim != 2 or min(arr.shape) < 1 or not np.isfinite(arr).all():
                    raise ValueError(f"Invalid or nonfinite tensor in {p.name}: {arr.shape}")
                arrays.append(arr)
                y.append(label)
                groups.append(group)
                source_classes.append(source_class)
                names.append(p.name)
                hashes.append(hashlib.sha256(p.read_bytes()).hexdigest())
            if cfg["groups_csv"]:
                mapping = pd.read_csv(cfg["groups_csv"], dtype=str, keep_default_na=False)
                if not {"filename", "group"}.issubset(mapping.columns):
                    raise ValueError("groups_csv must have columns: filename,group")
                if mapping["filename"].duplicated().any() or mapping["group"].str.strip().eq("").any():
                    raise ValueError("Group mapping has duplicate filenames or blank group identifiers.")
                lookup = mapping.set_index("filename")["group"].to_dict()
                missing = sorted(set(names)-set(lookup))
                if missing:
                    raise ValueError(f"Group mapping is missing {len(missing)} records, e.g. {missing[:3]}")
                groups = [lookup[name] for name in names]
                description = f"{cfg['group_level']}-disjoint (provided mapping)"
            else:
                if cfg["group_level"] == "subject":
                    raise ValueError("Subject-disjoint claims require groups_csv with verified subject IDs.")
                description = "filename-group-disjoint (subject identity not verified)"
                warnings.warn("Using filename-prefix groups after removing the complete class suffix. "
                              "These are not verified subject IDs. Supply groups_csv for subject-disjoint inference.")
            data = dict(arrays=arrays, y=np.asarray(y, dtype=int), groups=np.asarray(groups, dtype=str),
                        names=names, group_description=description, synthetic=False,
                        file_hashes=hashes, source_classes=source_classes)
        if set(np.unique(data["y"])) != {0, 1}:
            raise ValueError("Both classes are required: walking=1 and not walking=0. "
                             "Use explicit '-walking.parquet' and '-not_walking.parquet' labels; "
                             "the legacy '-standing.parquet' suffix is also class 0.")
        data["class_names"] = {0: "not walking", 1: "walking"}
        data["source_class_counts"] = dict(Counter(data["source_classes"]))
        data["label_counts"] = {"not walking (0)": int(np.sum(data["y"] == 0)),
                                "walking (1)": int(np.sum(data["y"] == 1))}
        print("Loaded class labels: " + "; ".join(
            f"{name} = {count}" for name, count in data["label_counts"].items()), flush=True)
        rows = {a.shape[0] for a in data["arrays"]}
        if len(rows) != 1:
            raise ValueError("Tensors have inconsistent row counts; fix upstream sensor layout before ablation.")
        return attach_duplicate_groups(data)



    def canonical_tensor_hash(array):
        """Exact numeric identity, including shape; +0 and -0 are equal numerically."""
        a = np.array(array, dtype="<f8", order="C", copy=True)
        a[a == 0.0] = 0.0
        h = hashlib.sha256()
        h.update(b"TQT-shape-aware-float64-v1\0")
        h.update(json.dumps(list(a.shape), separators=(",", ":")).encode("ascii"))
        h.update(b"\0")
        h.update(a.tobytes(order="C"))
        return h.hexdigest()


    def attach_duplicate_groups(data):
        """Retain all rows/labels; merge group constraints without consulting labels.

        Labels are read only AFTER connected components are fixed, for diagnostics.
        Common-prefix grouping is conservative: it prevents an exact crop/padding
        collision at any train-selected width, but may join some non-identical raw
        tensors. Those cases are reported separately, never called raw duplicates.
        """
        arrays, labels = data["arrays"], data["y"]
        original = np.asarray(data["groups"], dtype=str).copy()
        n = len(arrays)
        width = min(a.shape[1] for a in arrays)
        raw_keys = [canonical_tensor_hash(a) for a in arrays]
        prefix_keys = [canonical_tensor_hash(a[:, :width]) for a in arrays]
        parent = list(range(n))
        def find(i):
            while parent[i] != i:
                parent[i] = parent[parent[i]]
                i = parent[i]
            return i
        def union(i, j):
            a, b = find(i), find(j)
            if a != b:
                parent[max(a, b)] = min(a, b)
        def buckets(keys):
            result = {}
            for i, key in enumerate(keys):
                result.setdefault(str(key), []).append(i)
            return result
        raw_buckets, prefix_buckets = buckets(raw_keys), buckets(prefix_keys)
        # Original group constraints must never be broken, including supplied subjects.
        for by_key in (buckets(original), raw_buckets, prefix_buckets):
            for members in by_key.values():
                for i in members[1:]:
                    union(members[0], i)
        components = {}
        for i in range(n):
            components.setdefault(find(i), []).append(i)
        # Retain the existing group name where possible; merged groups use their
        # lexicographically smallest original group as an auditable representative.
        representatives = {root: min(original[i] for i in ids) for root, ids in components.items()}
        effective = np.asarray([representatives[find(i)] for i in range(n)], dtype=str)
        for by_key in (buckets(original), raw_buckets, prefix_buckets):
            assert all(len(set(effective[i] for i in ids)) == 1 for ids in by_key.values())
        data["original_groups"] = original
        data["groups"] = effective
        data["tensor_hashes"] = raw_keys
        data["prefix_hashes"] = prefix_keys
        data["original_group_description"] = data["group_description"]
        description = ("subject + duplicate-disjoint (provided IDs)" if "subject-disjoint (provided mapping)" in data["group_description"]
                       else "filename + duplicate-disjoint; subject identity unverified")
        if data["synthetic"]:
            description = "synthetic-group + duplicate-disjoint"
        elif "provided mapping" in data["group_description"] and "subject" not in data["group_description"]:
            description = "provided-group + duplicate-disjoint; subject identity unverified"
        data["group_description"] = description
        set_rows, record_rows = [], []
        for kind, by_key in (("raw_exact", raw_buckets), ("common_prefix", prefix_buckets)):
            for key, ids in by_key.items():
                if len(ids) < 2:
                    continue
                y = labels[ids]
                originals = sorted(set(original[ids]))
                item = dict(identity_kind=kind, identity_sha256=key, n_records=len(ids),
                    n_not_walking=int(np.sum(y == 0)), n_walking=int(np.sum(y == 1)),
                    mixed_labels=bool(len(np.unique(y)) > 1),
                    n_original_groups=len(originals), effective_group=str(effective[ids[0]]),
                    n_raw_identities=len({raw_keys[i] for i in ids}),
                    filenames=" | ".join(data["names"][i] for i in ids))
                set_rows.append(item)
        raw_sizes = {k: len(v) for k, v in raw_buckets.items()}
        pre_sizes = {k: len(v) for k, v in prefix_buckets.items()}
        raw_mixed = {k: len(set(labels[v])) > 1 for k, v in raw_buckets.items()}
        pre_mixed = {k: len(set(labels[v])) > 1 for k, v in prefix_buckets.items()}
        for i in range(n):
            record_rows.append(dict(sample_index=i, filename=data["names"][i], label=int(labels[i]),
                class_name="walking" if labels[i] else "not walking", source_class=data["source_classes"][i],
                original_group=str(original[i]), group=str(effective[i]),
                rows=int(arrays[i].shape[0]), columns=int(arrays[i].shape[1]),
                source_sha256=data["file_hashes"][i], tensor_sha256=raw_keys[i],
                raw_identity_records=raw_sizes[raw_keys[i]], raw_mixed_labels=raw_mixed[raw_keys[i]],
                prefix_sha256=prefix_keys[i], prefix_identity_records=pre_sizes[prefix_keys[i]],
                prefix_mixed_labels=pre_mixed[prefix_keys[i]]))
        data["manifest_rows"] = record_rows
        data["duplicate_set_rows"] = set_rows
        data["duplicate_record_rows"] = [r for r in record_rows
            if r["raw_identity_records"] > 1 or r["prefix_identity_records"] > 1]
        raw_sets = [r for r in set_rows if r["identity_kind"] == "raw_exact"]
        prefix_sets = [r for r in set_rows if r["identity_kind"] == "common_prefix"]
        data["cross_group_duplicate_tensors"] = sum(r["n_original_groups"] > 1 for r in raw_sets)
        audit = dict(policy="retain_all_labels_and_records; join original groups and duplicate constraints",
            n_loaded=n, n_retained=n, n_excluded=0, n_relabeled=0,
            n_original_groups=len(set(original)), n_effective_groups=len(set(effective)),
            raw_duplicate_sets=len(raw_sets), raw_duplicate_records=sum(r["n_records"] for r in raw_sets),
            raw_mixed_label_sets=sum(r["mixed_labels"] for r in raw_sets),
            raw_mixed_label_records=sum(r["n_records"] for r in raw_sets if r["mixed_labels"]),
            common_prefix_columns=int(width), prefix_duplicate_sets=len(prefix_sets),
            prefix_sets_with_distinct_raw_tensors=sum(r["n_raw_identities"] > 1 for r in prefix_sets),
            prefix_mixed_label_sets=sum(r["mixed_labels"] for r in prefix_sets),
            largest_effective_group=max(len(v) for v in components.values()),
            hash_definition="SHA256(shape + canonical little-endian float64 values); signed zero normalized",
            duplicate_groups_label_independent=True,
            grouping_rule="Connected components of original groups, raw identity, and shared-column-prefix identity",
            prefix_scope="Split constraint only, not a globally fitted feature transform",
            label_counts=dict(Counter("walking" if int(v) else "not walking" for v in labels)),
            interpretation="Mixed labels for identical features are retained observational ambiguity, not automatically corrected ground truth.")
        data["duplicate_audit"] = audit
        data["fingerprint"] = digest({"files": list(zip(data["names"], data["file_hashes"])),
            "tensor_hashes": raw_keys, "prefix_hashes": prefix_keys,
            "y": labels, "original_groups": original, "groups": effective,
            "duplicate_policy_version": "retain-group-v1"})
        print(f"Duplicate audit: {audit['raw_duplicate_sets']} exact-duplicate sets; "
              f"{audit['raw_mixed_label_sets']} with both labels; all {n} records retained.", flush=True)
        print(f"Effective split groups: {audit['n_original_groups']} -> {audit['n_effective_groups']}; "
              f"{audit['prefix_sets_with_distinct_raw_tensors']} additional common-prefix sets.", flush=True)
        return data


    class DataFeasibilityError(ValueError):
        """A declared study design cannot be evaluated on these data; export diagnostics."""


    def valid_split_candidates(y, groups, n_splits, seed, attempts, min_train):
        """Yield only group-disjoint, two-class splits; no model is fitted or scored."""
        y, groups = np.asarray(y), np.asarray(groups)
        if len(np.unique(groups)) < n_splits:
            return
        if any(len(np.unique(groups[y == cls])) < n_splits for cls in (0, 1)):
            return
        for attempt in range(int(attempts)):
            actual_seed = int((seed + attempt) % (2**32 - 1))
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", UserWarning)
                splits = list(StratifiedGroupKFold(n_splits=n_splits, shuffle=True,
                    random_state=actual_seed).split(np.zeros((len(y), 1)), y, groups))
            cover = np.zeros(len(y), dtype=int)
            valid = True
            for tr, va in splits:
                if (len(tr) < min_train or len(va) == 0 or
                    len(np.unique(y[tr])) != 2 or len(np.unique(y[va])) != 2):
                    valid = False
                    break
                if set(groups[tr]) & set(groups[va]):
                    raise AssertionError("A splitter violated a duplicate/group constraint.")
                cover[va] += 1
            if valid and np.all(cover == 1):
                yield splits, actual_seed


    def nested_split_plan(data, cfg):
        """Prefer requested folds; any fallback uses feasibility, never test scores."""
        requested = (int(cfg["outer_folds"]), int(cfg["inner_folds"]))
        outer_values = range(requested[0], 1, -1) if cfg["allow_fold_reduction"] else [requested[0]]
        inner_values = range(requested[1], 1, -1) if cfg["allow_fold_reduction"] else [requested[1]]
        nmin = int(cfg["n_components"])+1
        for no in outer_values:
            for ni in inner_values:
                for outer, outer_seed in valid_split_candidates(data["y"], data["groups"], no,
                        cfg["outer_seed"], cfg["split_search_attempts"], nmin):
                    inner_by_fold, inner_seeds = [], []
                    for fold, (tr, te) in enumerate(outer):
                        found = next(valid_split_candidates(data["y"][tr], data["groups"][tr], ni,
                            cfg["inner_seed"]+fold, cfg["split_search_attempts"], nmin), None)
                        if found is None:
                            break
                        inner, seed = found
                        inner_by_fold.append(inner)
                        inner_seeds.append(seed)
                    if len(inner_by_fold) != no:
                        continue
                    entries = []
                    for fold, ((tr, te), inner) in enumerate(zip(outer, inner_by_fold)):
                        for key in ("original_groups", "tensor_hashes", "prefix_hashes"):
                            ids = np.asarray(data[key])
                            assert not (set(ids[tr]) & set(ids[te])), f"Outer leakage in {key}"
                            for it, iv in inner:
                                assert not (set(ids[tr[it]]) & set(ids[tr[iv]])), f"Inner leakage in {key}"
                        entries.append(dict(fold=fold, outer_train=tr, outer_test=te,
                            inner=[dict(train=tr[it], validation=tr[iv]) for it, iv in inner]))
                    audit = dict(requested_outer_folds=requested[0], requested_inner_folds=requested[1],
                        actual_outer_folds=no, actual_inner_folds=ni, actual_outer_seed=outer_seed,
                        actual_inner_seeds=inner_seeds, requested_outer_seed=cfg["outer_seed"],
                        requested_inner_seed=cfg["inner_seed"],
                        selection="First feasible nested grouped split in deterministic order; no model scores",
                        leakage_checks="Original groups, raw tensor identities and common-prefix identities disjoint in ALL outer/inner splits",
                        rowwise_fallback=False, minimum_inner_training_rows=nmin)
                    if (no, ni) != requested or outer_seed != cfg["outer_seed"] or inner_seeds != [cfg["inner_seed"]+f for f in range(no)]:
                        print(f"Split feasibility adjustment: {no} outer / {ni} inner folds; "
                              "exact seeds and requested settings saved in split_audit.json.", flush=True)
                    cfg["outer_folds"], cfg["inner_folds"] = no, ni
                    return outer, inner_by_fold, entries, audit
        raise DataFeasibilityError(
            "No feasible nested grouped partition was found under the declared split search. "
            f"There are {len(data['y'])} records in {len(np.unique(data['groups']))} effective groups. "
            "Each train/evaluation partition must contain both labels; inner training needs at least "
            f"{nmin} rows for the fixed 16-component protocol. Original groups and identical tensors "
            "will not be separated merely to produce a score. All records and conflicts are listed in the audit.")


    class TensorContraction:
        """Train-only width selection, scalar normalization and 16-component PLS/PCA."""
        def __init__(self, kind, n_components=16):
            self.kind, self.n_components = kind, int(n_components)

        def _flatten(self, arrays):
            vectors = []
            for a in arrays:
                if a.shape[0] != self.rows:
                    raise ValueError("Inconsistent sensor rows.")
                cropped = a[:, :self.width]
                if cropped.shape[1] < self.width:
                    cropped = np.pad(cropped, ((0, 0), (0, self.width-cropped.shape[1])))
                vectors.append(cropped.reshape(-1))
            return np.stack(vectors).astype(np.float64)

        def fit(self, arrays, y):
            self.rows = arrays[0].shape[0]
            self.width = min(a.shape[1] for a in arrays)  # TRAINING tensors only.
            X = self._flatten(arrays)
            if self.n_components > min(X.shape[0]-1, X.shape[1]):
                raise ValueError(f"Cannot fit exactly {self.n_components} components on training shape {X.shape}.")
            self.mean = float(X.mean())
            self.std = float(X.std())
            if self.std < 1e-12:
                raise ValueError("Training tensors are constant.")
            X = (X-self.mean)/(self.std+1e-6)
            if self.kind == "pls":
                self.estimator = PLSRegression(n_components=self.n_components, scale=False,
                                               max_iter=500, tol=1e-6)
                with warnings.catch_warnings(record=True) as caught:
                    warnings.simplefilter("always")
                    self.estimator.fit(X, np.asarray(y, float).reshape(-1, 1))
                if any(issubclass(w.category, ConvergenceWarning) for w in caught):
                    raise RuntimeError("PLS failed to converge. Inspect training data; do not suppress the warning.")
                for w in caught:
                    warnings.warn(str(w.message))
            elif self.kind == "pca":
                self.estimator = PCA(n_components=self.n_components, svd_solver="full")
                self.estimator.fit(X)
            else:
                raise ValueError(self.kind)
            Z = np.asarray(self.estimator.transform(X), dtype=float)
            effective = int(np.sum(Z.std(axis=0) > 1e-10))
            if Z.shape != (len(arrays), self.n_components) or not np.isfinite(Z).all():
                raise DataFeasibilityError(f"{self.kind.upper()} did not return a finite 16-column transform.")
            self.effective_components = effective
            self.n_training_records = len(arrays)
            if effective < self.n_components:
                warnings.warn(f"{self.kind.upper()} returned {effective} variable columns out of "
                    f"{self.n_components}. All 16 output columns are retained; rank deficiency is logged.")
            return self

        def diagnostics(self):
            return dict(front_end=self.kind, requested_components=self.n_components,
                variable_output_columns=self.effective_components, n_training_records=self.n_training_records,
                training_rows=self.rows, training_width=self.width,
                zero_variance_columns_retained=self.n_components-self.effective_components)

        def transform(self, arrays):
            X = (self._flatten(arrays)-self.mean)/(self.std+1e-6)
            return np.asarray(self.estimator.transform(X), dtype=np.float64)


    def subset(data, idx):
        return [data["arrays"][int(i)] for i in idx]


    def prepare_contractions(data, outer_train, outer_test, inner_splits, kind, cfg):
        inner = []
        for itr, iva in inner_splits:
            tr, va = outer_train[itr], outer_train[iva]
            contraction = TensorContraction(kind, cfg["n_components"]).fit(subset(data, tr), data["y"][tr])
            inner.append(dict(Ztr=contraction.transform(subset(data, tr)),
                Zva=contraction.transform(subset(data, va)), ytr=data["y"][tr], yva=data["y"][va],
                diagnostics=contraction.diagnostics()))
        final = TensorContraction(kind, cfg["n_components"]).fit(subset(data, outer_train), data["y"][outer_train])
        return dict(inner=inner, Ztr=final.transform(subset(data, outer_train)),
            Zte=final.transform(subset(data, outer_test)), contraction=final,
            diagnostics=[dict(partition=f"inner_train_{i+1}", **p["diagnostics"]) for i, p in enumerate(inner)]
                + [dict(partition="outer_train", **final.diagnostics())])


    class BalancedHead:
        def __init__(self, C, cfg):
            self.cfg, self.C = cfg, float(C)
            self.scaler = StandardScaler() if cfg["standardize_head"] else None

        def fit(self, F, y):
            X = self.scaler.fit_transform(F) if self.scaler else np.asarray(F, float)
            for factor in (1, 5):
                self.model = LogisticRegression(C=self.C, solver="lbfgs", class_weight="balanced",
                    max_iter=int(self.cfg["lr_max_iter"]*factor), random_state=0)
                with warnings.catch_warnings(record=True) as caught:
                    warnings.simplefilter("always", ConvergenceWarning)
                    self.model.fit(X, y)
                failed = any(issubclass(w.category, ConvergenceWarning) for w in caught)
                for w in caught:
                    if not issubclass(w.category, ConvergenceWarning):
                        warnings.warn(str(w.message))
                if not failed:
                    return self
            raise RuntimeError("Logistic head did not converge after retry; no unreliable score was accepted.")

        def predict_proba(self, F):
            X = self.scaler.transform(F) if self.scaler else F
            return self.model.predict_proba(X)[:, 1]


    def select_score(y, p, cfg):
        pred = p >= .5
        if cfg["selection_metric"] == "accuracy":
            return float(np.mean(pred == y))
        return float(.5*(np.mean(pred[y == 0] == 0)+np.mean(pred[y == 1] == 1)))


    def model_specs(cfg):
        specs = [
            dict(id="majority", front="pls", feature="majority", learn="none"),
            dict(id="pls_lr", front="pls", feature="identity", learn="none"),
            dict(id="pca_lr", front="pca", feature="identity", learn="none"),
            dict(id="tanh_lr", front="pls", feature="pool", learn="none"),
            dict(id="tqt_fixed", front="pls", feature="notebook", learn="fixed"),
            dict(id="tqt_random", front="pls", feature="notebook", learn="random"),
            dict(id="tqt_trained", front="pls", feature="notebook", learn="cem"),
            dict(id="qt_exact", front="pls", feature="exact_nb", learn="cem"),
        ]
        if cfg["include_exact_grid_control"]:
            specs.append(dict(id="qt_exact_grid", front="pls", feature="exact_grid_nb", learn="cem"))
        if cfg["include_pca_tqt"]:
            specs.append(dict(id="pca_tqt", front="pca", feature="notebook", learn="cem"))
        if cfg["include_logical_pair"]:
            specs.extend([dict(id="logical_tqt", front="pls", feature="logical_tqt", learn="cem"),
                          dict(id="logical_qt", front="pls", feature="logical_qt", learn="cem")])
        return specs


    def get_features(Z, spec, engine, alpha, collect=False):
        if spec["feature"] == "identity":
            return Z, {}
        if spec["feature"] == "pool":
            return engine.pool(Z), {}
        return engine.features(Z, alpha, collect=collect)


    def inner_feature_pairs(parts, spec, engine, alpha):
        return [(get_features(p["Ztr"], spec, engine, alpha)[0], p["ytr"],
                 get_features(p["Zva"], spec, engine, alpha)[0], p["yva"])
                 for p in parts["inner"]]


    def score_inner_pairs(pairs, C, cfg):
        scores = []
        for Ftr, ytr, Fva, yva in pairs:
            head = BalancedHead(C, cfg).fit(Ftr, ytr)
            scores.append(select_score(yva, head.predict_proba(Fva), cfg))
        return float(np.mean(scores))


    def optimize_alpha(parts, spec, engine, seed, cfg):
        rng = np.random.RandomState(seed)
        mean = np.asarray(cfg["fixed_alpha"], float).copy()
        sigma = np.full(3, cfg["cem_sigma"], dtype=float)
        best_alpha, best_score = mean.copy(), -np.inf
        trace = []
        def evaluate(a, epoch, candidate):
            nonlocal best_alpha, best_score
            pairs = inner_feature_pairs(parts, spec, engine, a)
            score = score_inner_pairs(pairs, cfg["cem_head_C"], cfg)
            trace.append(dict(epoch=epoch, candidate=candidate, score=score,
                              alpha1=float(a[0]), alpha2=float(a[1]), alpha3=float(a[2])))
            if score > best_score:
                best_alpha, best_score = a.copy(), score
            return score
        # Include the fixed baseline in the search; this guarantees the initial
        # fixed point is considered without looking at any outer-test labels.
        evaluate(mean, 0, 0)
        for epoch in range(1, int(cfg["cem_epochs"])+1):
            noise = rng.randn(int(cfg["cem_popsize"]), 3)
            candidates = [a for z in noise for a in (mean+sigma*z, mean-sigma*z)]
            scores = np.array([evaluate(a, epoch, i) for i, a in enumerate(candidates)])
            order = np.argsort(-scores, kind="stable")
            elite = order[:max(1, len(order)//3)]
            mean = np.mean([candidates[i] for i in elite], axis=0)
            sigma *= cfg["cem_decay"]
            if epoch == 1 or epoch % 5 == 0 or epoch == cfg["cem_epochs"]:
                print(f"      CEM {epoch:02d}/{cfg['cem_epochs']}: inner {cfg['selection_metric']}={best_score:.4f}", flush=True)
        return best_alpha, best_score, trace


    def metric_values(y, p):
        pred = (p >= .5).astype(int)
        return dict(accuracy=float(accuracy_score(y, pred)),
            balanced_accuracy=float(balanced_accuracy_score(y, pred)),
            f1=float(f1_score(y, pred, zero_division=0)),
            mcc=float(matthews_corrcoef(y, pred)), roc_auc=float(roc_auc_score(y, p)),
            average_precision=float(average_precision_score(y, p)),
            log_loss=float(log_loss(y, p, labels=[0, 1])))


    def save_checkpoint(path, meta, p, pred, trace):
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        temp = path.with_suffix(".tmp")
        with temp.open("wb") as f:
            np.savez_compressed(f, probability=np.asarray(p), prediction=np.asarray(pred),
                metadata=np.asarray(json.dumps(meta, default=json_default, allow_nan=False)),
                trace=np.asarray(json.dumps(trace, default=json_default, allow_nan=False)))
        temp.replace(path)


    def load_checkpoint(path):
        with np.load(path, allow_pickle=False) as z:
            return json.loads(str(z["metadata"])), z["probability"].copy(), z["prediction"].copy(), json.loads(str(z["trace"]))


    def fit_one(data, tr, te, parts, spec, engine, seed, fold, cfg):
        started = time.perf_counter()
        trace = []
        alpha, C, inner_score = None, None, None
        ytr, yte = data["y"][tr], data["y"][te]
        if spec["feature"] == "majority":
            majority = int(ytr.mean() > .5)
            # This is a majority classifier, not a stratified or random dummy.
            p = np.full(len(te), float(majority))
            features, parameters, resources = 0, 0, {}
        else:
            if spec["learn"] == "cem":
                alpha, _, trace = optimize_alpha(parts, spec, engine,
                    int(seed+10000*fold), cfg)
            elif spec["learn"] == "random":
                # One predeclared draw per seed, same draw across folds. Never select
                # a best random realization, train it, or average probabilities.
                rng = np.random.RandomState(seed)
                alpha = np.asarray(cfg["fixed_alpha"])+cfg["cem_sigma"]*rng.randn(3)
            else:
                alpha = np.asarray(cfg["fixed_alpha"], float)
            pairs = inner_feature_pairs(parts, spec, engine, alpha)
            c_scores = [score_inner_pairs(pairs, c, cfg) for c in cfg["head_C_grid"]]
            chosen = int(np.argmax(c_scores))  # Stable, predeclared grid-order tie break.
            C, inner_score = float(cfg["head_C_grid"][chosen]), float(c_scores[chosen])
            Ftr, _ = get_features(parts["Ztr"], spec, engine, alpha)
            Fte, resources = get_features(parts["Zte"], spec, engine, alpha, collect=True)
            head = BalancedHead(C, cfg).fit(Ftr, ytr)
            p = head.predict_proba(Fte)
            features = int(Ftr.shape[1])
            parameters = features+1+(3 if spec["learn"] == "cem" else 0)
        pred = (p >= .5).astype(int)
        meta = dict(model=spec["id"], label=LABELS[spec["id"]], fold=int(fold), seed=int(seed),
            train_mode=spec["learn"], front_end=spec["front"], alpha=alpha,
            C=C, inner_score=inner_score, threshold=.5, feature_dimension=features,
            trainable_parameters_after_contraction=parameters,
            n_train=len(tr), n_test=len(te), n_test_groups=len(np.unique(data["groups"][te])),
            test_indices=np.asarray(te), metrics=metric_values(yte, p),
            fit_and_evaluation_seconds=float(time.perf_counter()-started), resources=resources,
            number_cem_candidates=len(trace), reused_deterministic_fit=False)
        return meta, p, pred, trace
    # ---------------------------------------------------------------------------
    # Paired OOF analysis. Seeds are repeated fits, NOT independent observations.
    # ---------------------------------------------------------------------------

    def count_scores(counts):
        tn, fp, fn, tp = np.moveaxis(np.asarray(counts, float), -1, 0)
        total = tn+fp+fn+tp
        acc = (tn+tp)/total
        ba = .5*(tp/(tp+fn)+tn/(tn+fp))
        f1_den = 2*tp+fp+fn
        f1 = np.divide(2*tp, f1_den, out=np.zeros_like(tp), where=f1_den > 0)
        den = np.sqrt((tp+fp)*(tp+fn)*(tn+fp)*(tn+fn))
        mcc = np.divide(tp*tn-fp*fn, den, out=np.zeros_like(tp), where=den > 0)
        return np.stack((acc, ba, f1, mcc), axis=-1)


    def contrast_specs(models):
        definitions = [
            ("pls_vs_pca", "PLS vs PCA with the same LR head", {"pls_lr": 1, "pca_lr": -1}),
            ("tqt_vs_pls", "Trained TQT-NB minus PLS + LR", {"tqt_trained": 1, "pls_lr": -1}),
            ("trained_vs_fixed", "Trained minus fixed TQT-NB", {"tqt_trained": 1, "tqt_fixed": -1}),
            ("trained_vs_random", "Trained minus random frozen TQT-NB", {"tqt_trained": 1, "tqt_random": -1}),
            ("nb_vs_exact", "TQT-NB minus exact notebook motifs", {"tqt_trained": 1, "qt_exact": -1}),
            ("nb_vs_exact_grid", "TQT-NB minus grid-matched exact motifs", {"tqt_trained": 1, "qt_exact_grid": -1}),
            ("pca_tqt_vs_pca", "TQT-NB increment after PCA", {"pca_tqt": 1, "pca_lr": -1}),
            ("front_end_interaction", "TQT increment: PLS minus PCA", {"tqt_trained": 1, "pls_lr": -1, "pca_tqt": -1, "pca_lr": 1}),
            ("logical_pair", "Repaired logical TQT minus logical QT", {"logical_tqt": 1, "logical_qt": -1}),
        ]
        return [(i, label, weights) for i, label, weights in definitions if set(weights).issubset(models)]


    def aggregate_results(data, records, specs, cfg):
        models = [s["id"] for s in specs]
        seeds = list(cfg["run_seeds"])
        N, M, S = len(data["y"]), len(models), len(seeds)
        probabilities = np.full((M, S, N), np.nan)
        predicted = np.full((M, S, N), -1, dtype=int)
        fold_of_row = np.full(N, -1, dtype=int)
        fold_rows, oof_rows, training_rows = [], [], []
        for meta, p, pred, trace in records:
            model, te = meta["model"], np.asarray(meta["test_indices"], dtype=int)
            mi = models.index(model)
            repeat_seeds = seeds if meta["train_mode"] not in ("cem", "random") else [meta["seed"]]
            for seed in repeat_seeds:
                si = seeds.index(seed)
                if np.isfinite(probabilities[mi, si, te]).any():
                    raise AssertionError("Duplicate OOF predictions for a model/seed/sample.")
                probabilities[mi, si, te] = p
                predicted[mi, si, te] = pred
                fold_of_row[te] = meta["fold"]
                fold_rows.append(dict(model=model, fold=meta["fold"], seed=seed,
                    reused_deterministic_fit=bool(seed != meta["seed"]),
                    feature_dimension=meta["feature_dimension"],
                    trainable_parameters_after_contraction=meta["trainable_parameters_after_contraction"],
                    C=meta["C"], inner_score=meta["inner_score"],
                    seconds=meta["fit_and_evaluation_seconds"], **meta["metrics"]))
            for t in trace:
                training_rows.append(dict(model=model, fold=meta["fold"], seed=meta["seed"], **t))
        if not np.isfinite(probabilities).all() or (predicted < 0).any() or (fold_of_row < 0).any():
            raise AssertionError("Incomplete paired OOF predictions; no partial manuscript report is allowed.")
        for mi, model in enumerate(models):
            for si, seed in enumerate(seeds):
                for i in range(N):
                    oof_rows.append(dict(model=model, seed=seed, fold=int(fold_of_row[i]),
                        sample_index=i, filename=data["names"][i], group=data["groups"][i],
                        original_group=data["original_groups"][i], tensor_sha256=data["tensor_hashes"][i],
                        true_label=int(data["y"][i]), probability=float(probabilities[mi, si, i]),
                        prediction=int(predicted[mi, si, i])))
        fold_df = pd.DataFrame(fold_rows)
        seed_df = pd.DataFrame([dict(model=model, seed=seed,
            **metric_values(data["y"], probabilities[mi, si]))
            for mi, model in enumerate(models) for si, seed in enumerate(seeds)])
        _, group_index = np.unique(data["groups"], return_inverse=True)
        G = int(group_index.max()+1)
        counts = np.zeros((M, S, G, 4), dtype=float)
        for mi in range(M):
            for si in range(S):
                codes = 2*data["y"]+predicted[mi, si]
                np.add.at(counts[mi, si], (group_index, codes), 1)
        central = count_scores(counts.sum(axis=2)).mean(axis=1)
        rng = np.random.RandomState(cfg["bootstrap_seed"])
        B = int(cfg["bootstrap_replicates"])
        bootstrap, attempts = [], 0
        while sum(len(x) for x in bootstrap) < B:
            size = min(200, B-sum(len(x) for x in bootstrap))
            weights = rng.multinomial(G, np.full(G, 1/G), size=size)
            sampled = np.einsum("bg,msgk->bmsk", weights, counts, optimize=True)
            good = (sampled[:, 0, 0, :2].sum(axis=-1) > 0) & (sampled[:, 0, 0, 2:].sum(axis=-1) > 0)
            if np.any(good):
                bootstrap.append(count_scores(sampled[good]).mean(axis=2))
            attempts += size
            if attempts > 20*B:
                raise RuntimeError("Too many bootstrap samples lack a class; more independent groups are needed.")
        bootstrap = np.concatenate(bootstrap, axis=0)[:B]
        low, high = np.quantile(bootstrap, [.025, .975], axis=0)
        summary_rows = []
        for mi, model in enumerate(models):
            row = dict(model=model, label=LABELS[model], n_outer_folds=cfg["outer_folds"],
                       n_run_seeds=S, n_resampling_groups=G,
                       feature_dimension=int(fold_df.loc[fold_df.model == model, "feature_dimension"].iloc[0]),
                       trainable_parameters_after_contraction=int(fold_df.loc[fold_df.model == model,
                                                        "trainable_parameters_after_contraction"].iloc[0]))
            for metric in METRICS:
                by_fold = fold_df.loc[fold_df.model == model].groupby("fold")[metric].mean()
                by_seed = seed_df.loc[seed_df.model == model, metric]
                row[f"{metric}_fold_mean"] = float(by_fold.mean())
                row[f"{metric}_fold_sd"] = float(by_fold.std(ddof=1))
                row[f"{metric}_oof_seed_mean"] = float(by_seed.mean())
                row[f"{metric}_oof_seed_sd"] = float(by_seed.std(ddof=1)) if S > 1 else 0.0
                if metric in COUNT_METRICS:
                    k = COUNT_METRICS.index(metric)
                    row[f"{metric}_ci_low"] = float(low[mi, k])
                    row[f"{metric}_ci_high"] = float(high[mi, k])
            summary_rows.append(row)
        contrast_rows = []
        for cid, label, weights in contrast_specs(models):
            w = np.array([weights.get(m, 0) for m in models], dtype=float)
            point = np.einsum("m,mk->k", w, central)
            dist = np.einsum("m,bmk->bk", w, bootstrap)
            lo, hi = np.quantile(dist, [.025, .975], axis=0)
            for k, metric in enumerate(COUNT_METRICS):
                contrast_rows.append(dict(contrast=cid, label=label, metric=metric,
                    estimate=float(point[k]), ci_low=float(lo[k]), ci_high=float(hi[k]),
                    units="proportion (multiply by 100 for percentage points)" if metric != "mcc" else "MCC units",
                    inference="Descriptive paired group-cluster bootstrap, conditional on fitted OOF models"))
        return dict(summary=pd.DataFrame(summary_rows), fold_metrics=fold_df,
            seed_metrics=seed_df, oof_predictions=pd.DataFrame(oof_rows),
            contrasts=pd.DataFrame(contrast_rows), training_trace=pd.DataFrame(training_rows),
            bootstrap=bootstrap, models=models, probabilities=probabilities,
            group_count=G, bootstrap_attempts=attempts)


    # ---------------------------------------------------------------------------
    # Manuscript figures, supporting PDF, CSVs, LaTeX and reviewer-result snippets.
    # All figures are vector PDF with 300-dpi raster fallback, plus 300-dpi PNG.
    # ---------------------------------------------------------------------------

    def text_pages(title, paragraphs, cfg, subtitle=""):
        lines = []
        for paragraph in paragraphs:
            lines.extend(textwrap.wrap(str(paragraph), width=98, break_long_words=False) or [""])
            lines.append("")
        while lines and not lines[-1].strip():
            lines.pop()
        figures = []
        for start in range(0, len(lines), 46):
            fig = plt.figure(figsize=(8.27, 11.69))
            fig.text(.065, .945, title+(" (continued)" if start else ""), fontsize=17, weight="bold", va="top")
            if subtitle:
                fig.text(.065, .900, subtitle, fontsize=9.5, va="top")
            body = "\n".join(lines[start:start+46])
            fig.text(.065, .864, body, fontsize=10, linespacing=1.5, va="top", family="DejaVu Sans")
            figures.append(fig)
        return figures


    def mark_synthetic(fig, cfg):
        if cfg["mode"] == "smoke":
            fig.text(.5, .992, "SYNTHETIC SMOKE TEST - NOT MANUSCRIPT RESULTS",
                     ha="center", va="top", fontsize=8, weight="bold")


    def accuracy_figure(summary, ids, title, cfg, group_description):
        table = summary.set_index("model").loc[ids]
        fig, ax = plt.subplots(figsize=(7.2, max(4.4, .48*len(ids)+1.55)))
        point = 100*table["accuracy_oof_seed_mean"].to_numpy()
        lo, hi = 100*table["accuracy_ci_low"].to_numpy(), 100*table["accuracy_ci_high"].to_numpy()
        ypos = np.arange(len(ids))
        ax.hlines(ypos, lo, hi, linewidth=1.6)
        ax.plot(point, ypos, "o", markersize=6)
        ax.set_yticks(ypos, [textwrap.fill(LABELS[m], 30) for m in ids], fontsize=8.7)
        ax.invert_yaxis()
        ax.set_ylim(len(ids)-.45, -.55)
        ax.set_xlim(max(0, float(lo.min())-4), min(100.5, float(hi.max())+4))
        ax.set_xlabel("Out-of-fold accuracy (%)")
        ax.grid(axis="x", linestyle=":", alpha=.35)
        ax.spines[["top", "right"]].set_visible(False)
        fig.suptitle(title, fontsize=13, weight="bold", y=.945)
        fig.text(.5, .88, f"{cfg['outer_folds']} paired outer folds | {len(cfg['run_seeds'])} run seeds | fixed epsilon = 12",
                 ha="center", fontsize=8.5)
        fig.subplots_adjust(left=.40, right=.97, top=.81, bottom=.23)
        foot = ("Points: mean of run-wise pooled OOF accuracies (not a probability ensemble). "
                "Bars: descriptive 95% group-cluster bootstrap intervals, conditional on fitted models. "
                "TQT-NB = audited notebook matrix implementation.")
        fig.text(.04, .13, textwrap.fill(foot, 112), fontsize=7.7, va="top")
        fig.text(.04, .035, group_description, fontsize=7.5)
        return fig


    def contrast_figure(contrasts, cfg):
        wanted = ["tqt_vs_pls", "trained_vs_fixed", "trained_vs_random", "nb_vs_exact", "nb_vs_exact_grid"]
        table = contrasts[(contrasts.metric == "accuracy") & contrasts.contrast.isin(wanted)].copy()
        table["_order"] = table.contrast.map({m: i for i, m in enumerate(wanted)})
        table = table.sort_values("_order")
        fig, ax = plt.subplots(figsize=(7.2, 4.6))
        y = np.arange(len(table))
        ax.axvline(0, linestyle="--", linewidth=1)
        ax.hlines(y, 100*table.ci_low, 100*table.ci_high, linewidth=1.6)
        ax.plot(100*table.estimate, y, "o", markersize=6)
        ax.set_yticks(y, [textwrap.fill(x, 31) for x in table.label], fontsize=8.7)
        ax.set_ylim(len(table)-.45, -.55)
        ax.set_xlabel("Paired accuracy difference (percentage points)")
        ax.grid(axis="x", linestyle=":", alpha=.35)
        ax.spines[["top", "right"]].set_visible(False)
        fig.suptitle("Incremental contribution of the trained TQT layer", fontsize=13, weight="bold", y=.94)
        fig.subplots_adjust(left=.42, right=.96, top=.81, bottom=.23)
        fig.text(.04, .13, textwrap.fill("Positive values favor the trained TQT-NB. Intervals use the same resampled groups "
            "for both arms. They are descriptive, conditional OOF intervals; seed repeats and overlapping CV "
            "training sets are not treated as independent experiments.", 111), fontsize=7.7, va="top")
        return fig


    def table_figure(summary, cfg):
        fig, ax = plt.subplots(figsize=(8.27, 7.5))
        ax.axis("off")
        rows = []
        for _, r in summary.iterrows():
            def fmt(metric, percent=False):
                scale = 100 if percent else 1
                digits = 1 if percent else 3
                return f"{r[metric+'_fold_mean']*scale:.{digits}f} +/- {r[metric+'_fold_sd']*scale:.{digits}f}"
            rows.append([textwrap.fill(r.label, 34), str(r.feature_dimension), fmt("accuracy", True),
                         fmt("balanced_accuracy", True), fmt("f1"), fmt("mcc")])
        table = ax.table(cellText=rows, colLabels=["Model", "Dim.", "Accuracy (%)", "Balanced acc. (%)", "F1", "MCC"],
            cellLoc="center", colWidths=[.36, .06, .145, .16, .14, .135], bbox=[0, .08, 1, .81])
        table.auto_set_font_size(False)
        table.set_fontsize(8)
        for (r, c), cell in table.get_celld().items():
            cell.visible_edges = "BT" if r == 0 else "B"
            cell.set_linewidth(.45 if r == 0 else .25)
            if r == 0:
                cell.set_text_props(weight="bold")
            if c == 0:
                cell.set_text_props(ha="left")
            cell.PAD = .06
        fig.suptitle("Ablation scorecard", fontsize=16, weight="bold", y=.955)
        fig.text(.065, .89, "Mean +/- SD across outer folds after averaging run-wise metrics within each fold.", fontsize=9)
        fig.text(.065, .083, textwrap.fill("Dim. counts features supplied to LR. PLS/PCA parameters are not included in head-parameter counts. "
            "F1 is for walking (label 1); threshold is fixed at 0.5. Logical-pair rows are a separate repaired sensitivity "
            "analysis, not a relabeling of the notebook model.", 115), fontsize=8, va="top")
        fig.subplots_adjust(left=.055, right=.965, bottom=.09, top=.92)
        return fig


    def fold_figure(result, cfg):
        ids = ["pls_lr", "tqt_fixed", "tqt_random", "tqt_trained", "qt_exact"]
        short = ["PLS + LR", "Fixed TQT-NB", "Random TQT-NB", "Trained TQT-NB", "Exact motifs"]
        fig, ax = plt.subplots(figsize=(7.2, 4.7))
        df = result["fold_metrics"].groupby(["model", "fold"])["accuracy"].mean()
        for fold in range(cfg["outer_folds"]):
            values = [100*df.loc[(m, fold)] for m in ids]
            ax.plot(np.arange(len(ids)), values, marker="o", label=f"Outer fold {fold+1}")
        ax.set_xticks(np.arange(len(ids)), short, rotation=15, ha="right", fontsize=9)
        ax.set_ylabel("Accuracy (%), averaged over run seeds")
        ax.set_title("Paired held-out-fold results", fontsize=13, weight="bold", pad=16)
        ax.grid(axis="y", linestyle=":", alpha=.35)
        ax.spines[["top", "right"]].set_visible(False)
        ax.legend(frameon=False, fontsize=8)
        fig.subplots_adjust(left=.12, right=.97, top=.83, bottom=.27)
        fig.text(.04, .07, "Connecting lines pair the same held-out groups. Only the outer folds enter the displayed fold SD.", fontsize=8)
        return fig


    def tex_escape(value):
        value = str(value)
        for old, new in [("&", r"\&"), ("%", r"\%"), ("_", r"\_"), ("#", r"\#")]:
            value = value.replace(old, new)
        return value


    def write_latex_and_response(result, data, cfg, out):
        s = result["summary"].set_index("model")
        contrasts = result["contrasts"]
        def val(model):
            r = s.loc[model]
            return f"{100*r.accuracy_fold_mean:.2f} \\pm {100*r.accuracy_fold_sd:.2f}"
        def delta(cid):
            r = contrasts[(contrasts.contrast == cid) & (contrasts.metric == "accuracy")].iloc[0]
            return f"{100*r.estimate:+.2f} percentage points (descriptive 95\\% interval [{100*r.ci_low:+.2f}, {100*r.ci_high:+.2f}])"
        rows = []
        for _, r in result["summary"].iterrows():
            row = [tex_escape(r.label), str(r.feature_dimension)]
            for metric in ("accuracy", "balanced_accuracy", "f1", "mcc"):
                scale, digits = (100, 2) if metric in ("accuracy", "balanced_accuracy") else (1, 3)
                row.append(f"${scale*r[metric+'_fold_mean']:.{digits}f} \\pm {scale*r[metric+'_fold_sd']:.{digits}f}$")
            rows.append(" & ".join(row)+r" \\")
        table = (r"\begin{table}[ht]"+"\n"+r"\centering\small"+"\n"+
            r"\caption{Matched ablations. Values are mean $\pm$ standard deviation across outer folds after averaging metrics over run seeds within each fold. F1 refers to walking. TQT-NB denotes the audited notebook matrix implementation; the logical pair is a separate repaired sensitivity analysis.}"+"\n"+
            r"\label{tab:r1_m1_ablation}"+"\n"+r"\begin{tabular}{lrrrrr}"+"\n"+r"\toprule"+"\n"+
            r"Model & Dim. & Accuracy (\%) & Balanced acc. (\%) & F1 & MCC \\"+"\n"+r"\midrule"+"\n"+
            "\n".join(rows)+"\n"+r"\bottomrule"+"\n"+r"\end{tabular}"+"\n"+r"\end{table}"+"\n")
        (out/"R1_M1_ablation_table.tex").write_text(table, encoding="utf-8")
        caption = ("Role of supervised preprocessing and TQT feature learning. All arms use the same "
            f"{cfg['outer_folds']} outer folds and group-disjoint inner selection. PLS has 16 components. "
            "The TQT arms have 54 features; their projection matrices, angle codebook and frozen compiler are shared. "
            "Fixed scales equal (0.8,0.8,0.8). Random scales are single predeclared draws and are never optimized or selected "
            "by their test performance. Circles show the mean of run-wise pooled out-of-fold accuracies, not an ensemble. "
            "Bars show descriptive 95\\% paired group-cluster bootstrap intervals conditional on the fitted OOF models. "
            "TQT-NB preserves the audited notebook matrices; exact notebook motifs replace the compiled one-qubit gates "
            "without changing the gate composition. The separate logical sensitivity pair is not shown here. "
            +tex_escape(data["group_description"])+".")
        qa = data["duplicate_audit"]
        caption += (f" All {qa['n_retained']} records and their original labels were retained, including "
            f"{qa['raw_mixed_label_records']} records in mixed-label exact-duplicate sets. Original groups and "
            "duplicate-connected groups were kept disjoint in every split. Common-prefix collisions were also protected.")
        (out/"R1_M1_figure_caption.tex").write_text(caption+"\n", encoding="utf-8")
        parts = [
            ("(a) Contribution of supervised PLS versus the TQT layer",
             "We refitted normalization and PLS within each inner-training partition and refitted them on the outer-training partition only after selection. "
             f"The PLS+LR and PCA+LR fold accuracies were ${val('pls_lr')}\\%$ and ${val('pca_lr')}\\%$, respectively. "
             f"The trained TQT-NB increment over PLS+LR in paired pooled OOF accuracy was {delta('tqt_vs_pls')}. "
             "These comparisons assess predictive increments and do not assign an additive fraction of the accuracy to PLS."),
            ("(b) PLS plus logistic regression without TQT",
             f"The direct PLS(16)+balanced logistic-regression baseline attained ${val('pls_lr')}\\%$ accuracy (fold mean $\\pm$ SD). "
             "It used the same groups, training-only preprocessing, inner-only regularization selection and fixed probability threshold of 0.5 as the feature-layer arms."),
            ("(c) Random or fixed TQT-feature comparison",
             f"Fixed and random-frozen TQT-NB features attained ${val('tqt_fixed')}\\%$ and ${val('tqt_random')}\\%$, respectively; "
             f"trained TQT-NB attained ${val('tqt_trained')}\\%$. "
             f"The paired trained-minus-fixed contrast was {delta('trained_vs_fixed')}, and the trained-minus-random contrast was {delta('trained_vs_random')}. "
             "All random realizations were retained; no best-seed selection or probability ensembling was performed."),
            ("(d) Comparison with the non-braided QT model",
             f"The architecture-matched exact-notebook-motif control attained ${val('qt_exact')}\\%$ accuracy. "
             f"The TQT-NB-minus-exact contrast was {delta('nb_vs_exact')}. "
             "This is a matched gate-substitution control for the current implementation, not the differently configured earlier-paper QT. "
             +(f"A separate repaired logical-CRY TQT/QT comparison gave {delta('logical_pair')}. " if cfg["include_logical_pair"] else "")+
             "The numerical audit identified phase-metric, braid-word replay and controlled-rotation composition discrepancies in the source. "
             "The notebook and repaired results must therefore be distinguished in any revised claims.")]
        text = "% Generated from actual ablation outputs; review interpretation and grouping before submission.\n"
        qa = data["duplicate_audit"]
        text += (f"% Data audit: retained={qa['n_retained']}; excluded=0; relabeled=0; "
                 f"mixed-label exact sets={qa['raw_mixed_label_sets']}; effective groups={qa['n_effective_groups']}.\n")
        if data["synthetic"]:
            text += "% SYNTHETIC SMOKE TEST -- DO NOT SUBMIT THESE VALUES.\n"
        for title, body in parts:
            text += "\\PARTRESPONSE{"+title+"}{\n"+body+"\n}\n\n"
        text += ("% Intervals are conditional on fitted OOF models, not full training-procedure uncertainty.\n"
                 "% epsilon=12 is historically chosen; these same-cohort results do not independently confirm optimality.\n")
        (out/"R1_M1_response_results.tex").write_text(text, encoding="utf-8")


    def generate_report(result, data, cfg, audit, self_tests, out):
        out = Path(out)
        for key in ("summary", "fold_metrics", "seed_metrics", "oof_predictions", "contrasts", "training_trace"):
            result[key].to_csv(out/f"{key}.csv", index=False)
        write_latex_and_response(result, data, cfg, out)
        with matplotlib.rc_context({"font.family": "DejaVu Sans", "font.size": 9,
            "pdf.fonttype": 42, "ps.fonttype": 42, "svg.fonttype": "none",
            "savefig.dpi": cfg["dpi"], "text.usetex": False}):
            main = accuracy_figure(result["summary"], ["pls_lr", "tqt_fixed", "tqt_random", "tqt_trained", "qt_exact", "pca_lr"],
                "PLS and TQT feature-learning ablation", cfg, data["group_description"])
            contrast = contrast_figure(result["contrasts"], cfg)
            foldplot = fold_figure(result, cfg)
            scorecard = table_figure(result["summary"], cfg)
            plots = [("R1_M1_ablation_figure", main), ("R1_M1_paired_differences", contrast),
                     ("R1_M1_fold_results", foldplot), ("R1_M1_scorecard", scorecard)]
            if cfg["include_logical_pair"]:
                logical = accuracy_figure(result["summary"], ["logical_tqt", "logical_qt"],
                    "Separate repaired logical-TQT / QT sensitivity", cfg, data["group_description"])
                plots.append(("R1_M1_logical_sensitivity", logical))
            for name, fig in plots:
                mark_synthetic(fig, cfg)
                fig.savefig(out/f"{name}.pdf", dpi=cfg["dpi"], bbox_inches="tight")
                fig.savefig(out/f"{name}.png", dpi=cfg["dpi"], bbox_inches="tight")
                fig.savefig(out/f"{name}.svg", dpi=cfg["dpi"], bbox_inches="tight")
            n0, n1 = np.bincount(data["y"], minlength=2)
            method_paragraphs = [
                f"Data: {len(data['y'])} tensors; {result['group_count']} groups; not walking=0: {n0}, walking=1: {n1}. "
                f"Validation: {data['group_description']}. Cross-group identical-tensor patterns: {data['cross_group_duplicate_tensors']}.",
                "IMPORTANT IMPLEMENTATION SCOPE: TQT-NB is the supplied notebook's matrix implementation. The numerical audit "
                "finds a non-phase-invariant compiler objective, non-replaying emitted words, and incorrect gate decompositions. "
                "The main ablation retains those matrices; repaired logical-pair results are separately labeled. "
                "These figures must not silently substantiate physical braid-program or CRY claims about the uncorrected model.",
                f"Protocol: {cfg['outer_folds']} stratified group-disjoint outer folds (actual seeds in split_audit.json); "
                f"{cfg['inner_folds']} stratified group-disjoint inner folds. Input width, global scalar normalization, PLS/PCA, "
                "and LR feature scaling are fitted on the applicable training partition only. Outer-test labels are used only "
                "for stratification and the final held-out metrics, never for tuning, threshold selection, early stopping, or seed selection.",
                f"Model: PLS(16), scale=False; parallel 16->8 / 16->6 / 16->4 pooling; codebook {cfg['codebook']}; "
                "54 post-bias expectation features. The old paper's image/QNN experiments are not used. "
                "Tanh-only pooling and PCA arms help locate the contribution of non-quantum operations.",
                f"Learning: mirrored CEM, {cfg['cem_epochs']} epochs, {cfg['cem_popsize']} mirrored pairs/epoch, "
                f"sigma={cfg['cem_sigma']}, decay={cfg['cem_decay']}; one additional evaluation of the fixed starting point. "
                f"Run seeds: {cfg['run_seeds']}. Selection metric: {cfg['selection_metric']}. CEM uses LR C={cfg['cem_head_C']}; "
                f"after scale selection, every arm selects C from {cfg['head_C_grid']} using the inner folds. "
                f"LR class_weight='balanced'; feature standardization={cfg['standardize_head']}; threshold=0.5. "
                "The three source scales are unconstrained, including possible negative values.",
                "Random TQT: one independent Gaussian scale draw per predeclared run seed, with the same starting mean and sigma "
                "as CEM. The draw is frozen across the entire fit, and its projection/codebook/compiler match the learned arm. "
                "Fixed TQT: alpha=(0.8,0.8,0.8). Only the balanced LR head is fitted in frozen controls.",
                "Compiler: fixed epsilon=12, canonical pi/128 angle-bin centers, deterministic per-key RNG, "
                f"dictionary depth {cfg['compiler_dictionary_depth']}, {cfg['compiler_trials']} sampled word pairs. "
                "Compilation does not fit sensor data or labels. Exact-motif controls share topology, pooling, readouts, "
                "scale search and head protocol. Exact-grid additionally shares angle rounding. "
                "The repaired logical pair corrects the phase objective and word replay and uses an exact logical CRY(pi/3).",
                "Statistics: table entries are fold mean +/- SD after averaging seed-specific scores within each fold. "
                "Plot centers and contrasts use pooled OOF metrics averaged over runs, not averaged probabilities. "
                f"Intervals use {cfg['bootstrap_replicates']} paired cluster bootstrap replicates of the {result['group_count']} groups. "
                "They are descriptive and conditional on fitted OOF models; they do not capture full retraining uncertainty "
                "or remove dependence from overlapping CV training sets. Seeds do not increase the number of independent groups. "
                "No significance stars or independent-window p-values are generated.",
                "Historical choice: epsilon=12 is inherited from the previously examined test sweep. Keeping it fixed prevents "
                "additional test selection here but does not turn this cohort into an independent confirmatory evaluation. "
                "Do not report the old mean-best accuracy as the new ablation result.",
            ]
            method_paragraphs += data_quality_paragraphs(data)
            method_paragraphs.append("Exact split configuration is in split_audit.json. "
                "Requested folds/seeds may be adjusted for grouped feasibility only, never using model scores. "
                "contraction_diagnostics.csv lists actual training-only widths and variable output columns; "
                "finite 16-column transforms are retained even when some columns are constant.")
            if result["group_count"] < 20:
                method_paragraphs.append("CAUTION: There are fewer than 20 independent resampling groups; bootstrap intervals may be unstable.")
            methods = text_pages("R1.M.1 | Ablation protocol and scope", method_paragraphs, cfg,
                                 "Matched preprocessing, feature-learning and non-braided controls")
            audit_paragraphs = [f"{k}: {v:.6g}" for k, v in audit["numerical_checks"].items()]
            audit_paragraphs += audit["interpretation_notes"]
            audit_paragraphs.append("Validation self-tests: "+json.dumps(self_tests))
            audit_pages = text_pages("Implementation audit", audit_paragraphs, cfg)
            figures = methods+[scorecard, main, contrast, foldplot]+([logical] if cfg["include_logical_pair"] else [])+audit_pages
            with PdfPages(out/"R1_M1_ablation_report.pdf", metadata={"Title": "R1.M.1 PLS/TQT ablation report",
                    "Subject": "Paired nested-group ablations with implementation audit", "Creator": VERSION}) as pdf:
                for page, fig in enumerate(figures, 1):
                    if fig not in [p[1] for p in plots]:
                        mark_synthetic(fig, cfg)
                    # Footer is outside plot/table regions and belongs to the report only.
                    fig.text(.975, .012, f"{page}/{len(figures)}", ha="right", fontsize=7)
                    pdf.savefig(fig, dpi=cfg["dpi"])
                    fig.savefig(out/f"R1_M1_ablation_report_page_{page:02d}.svg", dpi=cfg["dpi"])
            for fig in figures:
                if cfg["show_figures"]:
                    plt.figure(fig.number)
                    plt.show()
                plt.close(fig)
        return out/"R1_M1_ablation_report.pdf"
    # ---------------------------------------------------------------------------
    # Standalone runner, provenance, resumption and artifact export.
    # ---------------------------------------------------------------------------

    def data_quality_paragraphs(data):
        a = data["duplicate_audit"]
        return [
            f"Loaded {a['n_loaded']} tensors; retained {a['n_retained']}; excluded {a['n_excluded']}; relabeled {a['n_relabeled']}. "
            f"Recorded classes: {data['label_counts']}. Only Parquet tensors are model inputs; PGM previews are not loaded.",
            f"Exact full-tensor duplicate sets: {a['raw_duplicate_sets']}, containing {a['raw_duplicate_records']} records. "
            f"Mixed-label exact sets: {a['raw_mixed_label_sets']} ({a['raw_mixed_label_records']} records). "
            "The original labels are retained. Identical features with different recorded labels are an ambiguity to report, "
            "not a reason to silently choose one label or discard inconvenient records.",
            "Exact identity includes shape and canonical numeric values. This avoids calling arrays with equal bytes "
            "but different shapes identical. Common-prefix collisions are counted separately.",
            f"Original grouping units: {a['n_original_groups']}; effective connected-component groups: {a['n_effective_groups']}; "
            f"largest effective group: {a['largest_effective_group']} records. "
            "Every original group and every duplicate identity stays within one fold at all outer/inner levels. "
            "Merge decisions use features and original group IDs, not labels or model scores.",
            f"Common-prefix duplicate protection uses the {a['common_prefix_columns']} columns shared by all tensors "
            f"ONLY as a split constraint. {a['prefix_sets_with_distinct_raw_tensors']} prefix groups contain distinct full tensors. "
            "This conservative rule also prevents exact collisions after the existing training-width crop/pad operation. "
            "Actual tensor width, normalization, PLS/PCA and head scaling remain fitted using the applicable training partition only.",
            "Group interpretation: " + data["group_description"] + ". Filename-derived segment groups do not establish "
            "subject independence or rule out temporal overlap between non-identical windows.",
            "Files: dataset_manifest.csv contains every record, original/effective group IDs, labels, shapes and hashes. "
            "duplicate_tensor_audit.csv lists records affected by full-tensor or common-prefix identity. "
            "duplicate_sets.csv lists complete filenames and class counts per repeated identity. No data file is changed.",
        ]


    def write_data_audits(data, cfg, out):
        manifest = pd.DataFrame(data["manifest_rows"])
        manifest.to_csv(out/"dataset_manifest.csv", index=False)
        pd.DataFrame(data["duplicate_record_rows"], columns=manifest.columns).to_csv(out/"duplicate_tensor_audit.csv", index=False)
        columns = ["identity_kind", "identity_sha256", "n_records", "n_not_walking", "n_walking",
                   "mixed_labels", "n_original_groups", "effective_group", "n_raw_identities", "filenames"]
        pd.DataFrame(data["duplicate_set_rows"], columns=columns).to_csv(out/"duplicate_sets.csv", index=False)
        audit = dict(data_dir=cfg["data_dir"], label_counts=data["label_counts"],
            source_class_counts=data["source_class_counts"], label_mapping={"walking": 1, "not_walking": 0, "standing": 0},
            accepted_negative_aliases=["not-walking", "not walking"],
            matching="Explicit full class suffix; negative suffixes first; case-insensitive suffix",
            ignored_files="PGM previews, other non-Parquet files, hidden dotfiles including macOS AppleDouble",
            original_group_description=data["original_group_description"], group_description=data["group_description"],
            groups_csv=cfg["groups_csv"], synthetic=data["synthetic"], duplicates=data["duplicate_audit"])
        write_json(out/"data_loading_audit.json", audit)
        return audit


    def write_data_quality_report(data, cfg, out, status, detail, split_audit=None):
        paragraphs = ["STATUS: " + status, detail] + data_quality_paragraphs(data)
        if split_audit:
            paragraphs += ["Nested split audit: " + json.dumps(split_audit)]
        if status != "study_completed":
            paragraphs.append("This data-audit PDF contains no empirical ablation claims. "
                              "Only a completed ablation report contains evaluated model results.")
        with matplotlib.rc_context({"font.family": "DejaVu Sans", "font.size": 9,
                "pdf.fonttype": 42, "svg.fonttype": "none", "text.usetex": False}):
            figures = text_pages("R1.M.1 | Data quality and split audit", paragraphs, cfg,
                                 "All labels retained; duplicate identities protected across folds")
            pdf_path = out/"R1_M1_data_quality_report.pdf"
            with PdfPages(pdf_path, metadata={"Title": "R1.M.1 data quality audit", "Creator": VERSION}) as pdf:
                for page, fig in enumerate(figures, 1):
                    mark_synthetic(fig, cfg)
                    fig.text(.97, .015, f"{page}/{len(figures)}", ha="right", fontsize=7)
                    pdf.savefig(fig, dpi=cfg["dpi"])
                    fig.savefig(out/f"R1_M1_data_quality_page_{page:02d}.svg", dpi=cfg["dpi"])
                    plt.close(fig)
        return pdf_path


    def diagnostic_result(data, cfg, out, reason, status="diagnostic_only", split_audit=None):
        pdf = write_data_quality_report(data, cfg, out, status, reason, split_audit)
        write_json(out/"preflight_status.json", dict(status=status, reason=reason, model_results_generated=False))
        manifest_path = out/"run_manifest.json"
        manifest = json.loads(manifest_path.read_text(encoding="utf-8")) if manifest_path.exists() else {}
        manifest.update(completed=False, status=status, reason=reason, config=cfg)
        write_json(manifest_path, manifest)
        return dict(completed=False, status=status, reason=reason, output_dir=out,
                    report_pdf=pdf, config=cfg, data_audit=data["duplicate_audit"])


    def validate_config(cfg):
        if cfg["mode"] not in ("study", "smoke"):
            raise ValueError("mode must be 'study' or 'smoke'.")
        if cfg["n_components"] != 16 or cfg["epsilon"] != 12:
            raise ValueError("This matched R1.M.1 experiment fixes 16 components and epsilon=12. "
                             "A different contraction or epsilon sweep is a different protocol.")
        if cfg["selection_metric"] not in ("accuracy", "balanced_accuracy"):
            raise ValueError("selection_metric must be accuracy or balanced_accuracy.")
        for k in ("outer_folds", "inner_folds"):
            if int(cfg[k]) < 2:
                raise ValueError(f"{k} must be at least 2.")
        if not cfg["run_seeds"] or len(set(cfg["run_seeds"])) != len(cfg["run_seeds"]):
            raise ValueError("run_seeds must be nonempty and unique.")
        for key in ("head_C_grid", "codebook"):
            if not cfg[key] or not np.isfinite(cfg[key]).all() or np.min(cfg[key]) <= 0:
                raise ValueError(f"{key} must contain finite positive values.")
        if len(cfg["fixed_alpha"]) != 3 or not np.isfinite(cfg["fixed_alpha"]).all():
            raise ValueError("fixed_alpha must have three finite values.")
        if not (cfg["cem_sigma"] > 0 and 0 < cfg["cem_decay"] <= 1 and cfg["cem_head_C"] > 0):
            raise ValueError("Invalid CEM parameters.")
        if cfg["cem_epochs"] < 1 or cfg["cem_popsize"] < 1 or cfg["bootstrap_replicates"] < 20:
            raise ValueError("Need positive CEM budget and at least 20 bootstrap replicates.")
        if cfg["compiler_dictionary_depth"] < 1 or cfg["compiler_trials"] < 1 or cfg["angle_quantum"] <= 0:
            raise ValueError("Invalid compiler settings.")
        if cfg["dpi"] < 300:
            raise ValueError("Use dpi>=300 for manuscript output.")
        if int(cfg["split_search_attempts"]) < 1:
            raise ValueError("split_search_attempts must be positive.")


    def implementation_fingerprint():
        """Stable identity of the full worker AST, supplied by the stdlib launcher."""
        return _R1M1_ENGINE_SHA256


    def run(user_config=None):
        cfg = copy.deepcopy(CONFIG)
        if user_config is not None:
            unknown = set(user_config)-set(cfg)
            if unknown:
                raise ValueError(f"Unknown configuration keys: {sorted(unknown)}")
            cfg.update(copy.deepcopy(user_config))
        if cfg["mode"] == "smoke":
            cfg.update(cem_epochs=1, cem_popsize=2, run_seeds=(54, 1054),
                compiler_dictionary_depth=4, compiler_trials=64, bootstrap_replicates=100)
            cfg["output_dir"] = str(cfg["output_dir"])+"_SMOKE_TEST"
            print("SYNTHETIC SMOKE TEST: not study data, not manuscript results.", flush=True)
        validate_config(cfg)
        provenance = check_notebook_provenance(cfg)
        data = load_data(cfg)
        environment = {"python": sys.version, "platform": platform.platform()}
        for package in ("numpy", "pandas", "scipy", "scikit-learn", "matplotlib", "pyarrow", "threadpoolctl"):
            try:
                environment[package] = importlib.metadata.version(package)
            except importlib.metadata.PackageNotFoundError:
                environment[package] = "not installed"
        affecting = {k: v for k, v in cfg.items()
                    if k not in ("output_dir", "resume", "show_figures", "data_dir", "notebook_path", "groups_csv")}
        fingerprint = digest(dict(version=VERSION, implementation=implementation_fingerprint(),
            config=affecting, data=data["fingerprint"], environment=environment))
        out = Path(cfg["output_dir"])/("run_"+fingerprint[:12])
        out.mkdir(parents=True, exist_ok=True)
        (out/"checkpoints").mkdir(exist_ok=True)
        write_json(out/"run_manifest.json", dict(fingerprint=fingerprint, version=VERSION,
            config=cfg, environment=environment, provenance=provenance,
            dataset_fingerprint=data["fingerprint"], synthetic=data["synthetic"],
            group_description=data["group_description"], completed=False))
        write_data_audits(data, cfg, out)
        print(f"Output directory: {out.resolve()}", flush=True)
        write_data_quality_report(data, cfg, out, "preflight_running",
                                  "Duplicate checks completed. Nested splits and front ends are checked next.")
        try:
            outer, inner_by_fold, split_manifest, split_audit = nested_split_plan(data, cfg)
        except DataFeasibilityError as exc:
            return diagnostic_result(data, cfg, out, str(exc))
        write_json(out/"split_manifest.json", split_manifest)
        write_json(out/"split_audit.json", split_audit)
        print(f"Data: {len(data['y'])} records; {len(np.unique(data['groups']))} effective groups; "
              f"{data['group_description']}", flush=True)
        specs, records = model_specs(cfg), []
        prepared, front_diagnostics = {}, []
        print("Checking every inner/outer PLS and PCA fit before quantum compilation...", flush=True)
        try:
            with threadpool_limits(limits=cfg["blas_threads"]):
                for fold, (tr, te) in enumerate(outer):
                    prepared[fold] = {}
                    for front in sorted({s["front"] for s in specs if s["feature"] != "majority"}):
                        print(f"  Preflight fold {fold+1}/{cfg['outer_folds']}: {front.upper()}", flush=True)
                        parts = prepare_contractions(data, tr, te, inner_by_fold[fold], front, cfg)
                        prepared[fold][front] = parts
                        front_diagnostics.extend(dict(fold=fold, **row) for row in parts["diagnostics"])
        except (ValueError, RuntimeError, np.linalg.LinAlgError) as exc:
            pd.DataFrame(front_diagnostics).to_csv(out/"contraction_diagnostics.csv", index=False)
            return diagnostic_result(data, cfg, out,
                f"Front-end preflight could not complete the declared 16-component protocol: {type(exc).__name__}: {exc}. "
                "No TQT training or ablation scores were generated.", split_audit=split_audit)
        pd.DataFrame(front_diagnostics).to_csv(out/"contraction_diagnostics.csv", index=False)
        write_json(out/"preflight_status.json", dict(status="passed", effective_config=cfg, split_audit=split_audit))
        write_data_quality_report(data, cfg, out, "preflight_passed",
            "All grouped nested splits and all 16-column PLS/PCA transformations passed preflight. "
            "Exact seeds and retained zero-variance columns are documented in the CSV/JSON audits.", split_audit)
        if cfg["preflight_only"]:
            return diagnostic_result(data, cfg, out, "All data, grouped nested splits, and front-end transformations passed preflight. "
                "No model training was requested (preflight_only=True).", status="preflight_only", split_audit=split_audit)
        with threadpool_limits(limits=cfg["blas_threads"]):
            print("Checking compiler, braid replay and exact gate definitions...", flush=True)
            compiler = FrozenCompiler(cfg)
            audit = implementation_audit(cfg, compiler)
            tests = numerical_self_tests(cfg, compiler)
            write_json(out/"implementation_audit.json", audit)
            write_json(out/"numerical_self_tests.json", tests)
            print(f"Vectorized/source feature discrepancy: {tests['maximum_batch_vs_source_feature_error']:.3g}", flush=True)
            print("Source gate/compiler discrepancies are recorded in implementation_audit.json. "
                  "Notebook and repaired logical models are kept separate.", flush=True)
            engines = {}
            for kind in {s["feature"] for s in specs} - {"identity", "majority", "pool"}:
                engines[kind] = FeatureEngine(cfg, compiler, kind)
            engines["pool"] = engines["notebook"]
            for fold, (tr, te) in enumerate(outer):
                print(f"\nOUTER FOLD {fold+1}/{cfg['outer_folds']} | train {len(tr)} | test {len(te)}", flush=True)
                # Inner contractions are reused across all matched arms, not refitted
                # from a different split or accidentally replaced by outer-fitted PLS.
                parts_by_front = prepared.pop(fold)
                for spec in specs:
                    front = spec["front"]
                    seeds = cfg["run_seeds"] if spec["learn"] in ("cem", "random") else [cfg["run_seeds"][0]]
                    for seed in seeds:
                        checkpoint = out/"checkpoints"/f"fold{fold}_{spec['id']}_seed{seed}.npz"
                        if cfg["resume"] and checkpoint.exists():
                            record = load_checkpoint(checkpoint)
                            meta = record[0]
                            if meta.get("run_fingerprint") != fingerprint or meta["test_indices"] != te.tolist():
                                raise RuntimeError("Checkpoint fingerprint or held-out row IDs do not match.")
                            print(f"  Resumed {spec['id']} | seed {seed}", flush=True)
                        else:
                            front = spec["front"]
                            if spec["feature"] != "majority" and front not in parts_by_front:
                                print(f"  Fitting training-only {front.upper()} contractions...", flush=True)
                                parts_by_front[front] = prepare_contractions(data, tr, te, inner_by_fold[fold], front, cfg)
                            print(f"  {LABELS[spec['id']]} | seed {seed}", flush=True)
                            record = fit_one(data, tr, te, parts_by_front.get(front), spec,
                                engines.get(spec["feature"]), seed, fold, cfg)
                            record[0]["run_fingerprint"] = fingerprint
                            save_checkpoint(checkpoint, *record)
                            print(f"    held-out accuracy={record[0]['metrics']['accuracy']:.4f}; "
                                  f"balanced accuracy={record[0]['metrics']['balanced_accuracy']:.4f}", flush=True)
                        records.append(record)
                # A fold's fitted contractions need not remain in memory.
                del parts_by_front
            # Preserve all prior entries on a fully/partially resumed run. The
            # immutable per-key map makes merges independent of model evaluation order.
            inventory_path = out/"braid_inventory.jsonl"
            previous = {}
            if inventory_path.exists():
                with inventory_path.open(encoding="utf-8") as handle:
                    for line in handle:
                        row = json.loads(line)
                        previous[(row["mode"], row["axis"], row["qindex"])] = row
            compiler.inventory(out/"braid_inventory_current.jsonl")
            with (out/"braid_inventory_current.jsonl").open(encoding="utf-8") as handle:
                for line in handle:
                    row = json.loads(line)
                    previous[(row["mode"], row["axis"], row["qindex"])] = row
            with inventory_path.open("w", encoding="utf-8") as handle:
                for key in sorted(previous):
                    handle.write(json.dumps(previous[key])+"\n")
            (out/"braid_inventory_current.jsonl").unlink()
            print("\nAggregating paired OOF results and group-cluster uncertainty...", flush=True)
            result = aggregate_results(data, records, specs, cfg)
            write_json(out/"model_runs.json", [record[0] for record in records])
            pdf = generate_report(result, data, cfg, audit, tests, out)
            write_data_quality_report(data, cfg, out, "study_completed",
                "All ablation arms completed. Full results are in R1_M1_ablation_report.pdf; this document records the data constraints.",
                split_audit)
        # Final completion is recorded only after every model, prediction and PDF exists.
        write_json(out/"run_manifest.json", dict(fingerprint=fingerprint, version=VERSION,
            config=cfg, environment=environment, provenance=provenance,
            dataset_fingerprint=data["fingerprint"], synthetic=data["synthetic"],
            group_description=data["group_description"], completed=True, status="completed",
            duplicate_audit=data["duplicate_audit"], split_audit=split_audit,
            group_bootstrap_replicates=cfg["bootstrap_replicates"],
            group_bootstrap_attempts=result["bootstrap_attempts"],
            inference_scope="Descriptive conditional OOF intervals; seeds are not independent groups."))
        print("\nCompleted. No historical accuracies were substituted.", flush=True)
        print(f"Manuscript figure: {(out/'R1_M1_ablation_figure.pdf').resolve()}")
        print(f"Paired differences: {(out/'R1_M1_paired_differences.pdf').resolve()}")
        print(f"Full report: {pdf.resolve()}")
        print(f"LaTeX table and response-result snippets: {out.resolve()}")
        print(result["summary"][["model", "accuracy_fold_mean", "accuracy_fold_sd",
                                  "balanced_accuracy_fold_mean"]].to_string(index=False))
        if data["synthetic"]:
            print("SYNTHETIC SMOKE TEST ONLY. Do not put these values or figures in a manuscript.")
        result.update(output_dir=out, report_pdf=pdf, config=cfg, audit=audit, self_tests=tests,
                      completed=True, status="completed", data_audit=data["duplicate_audit"])
        return result


    return run(CONFIG)


if __name__ == "__main__":
    RESULTS = run(CONFIG)


Data directory: /Users/h4/Desktop/Doctorado Univ Rioja/Papers/TQC/outs
Inputs: Parquet tensors only; PGM previews ignored. Duplicate-label records will be retained and grouped, not rejected.
Current environment failed its dependency check: ImportError: numpy.core.multiarray failed to import
Using a separate compatible environment; no packages in the notebook kernel will be changed.
Reusing compatible environment: /Users/h4/.cache/tqt_r1m1/env_py312_ddcd85dcea7ade05
Running complete ablation implementation with: /Users/h4/.cache/tqt_r1m1/env_py312_ddcd85dcea7ade05/bin/python
Execution log: /Users/h4/Desktop/Doctorado Univ Rioja/Papers/TQC/outputs/R1_M1_ablation/_launches/20260907_112844_ee547414/analysis.log
/Users/h4/.cache/tqt_r1m1/workers/r1m1_worker_5ab4f27f33f9cdaaded5.py:799: UserWarning: Using filename-prefix groups after removing the complete class suffix. These are not verified subject IDs. Supply groups_csv for subject-disjoint inference.
  warnings.warn("Using filename-prefix